# Fashion RAG Chatbot - ViFashionCLIP + BGE-M3 Multimodal

Notebook nay trien khai chatbot tu van thoi trang da phuong thuc (text + image), su dung kien truc RAG hai lop (Layer A + Layer B) ket hop ViFashionCLIP tieng Viet.

---

## Kien truc he thong

```text
USER INPUT (text / image)
  -> VALIDATE INPUT
  -> IMAGE PROCESSING neu co anh
       PERSON  -> analyze_person_image -> user_profile
       PRODUCT -> search_products_by_image hoac caption_product_image
  -> INTENT DETECTION
       GREETING / CHITCHAT / SEARCH / OUTFIT / IMAGE_SEARCH
  -> RAG / OUTFIT / IMAGE response
  -> LLM RESPONSE
```

---

## Hai lop Embedding

| Layer | Muc dich | Model | Dim |
|-------|----------|-------|-----|
| Layer A text | Tim kiem san pham qua text | ViFashionCLIPTextEmbeddings | 512 |
| Layer A image | Tim kiem san pham qua anh | FashionCLIPImageEmbeddings | 512 |
| Layer B | Quy tac phoi do | BGEM3Embeddings | 1024 |

Layer A va Layer B dung collection Qdrant rieng biet do vector dim khac nhau.

---

## Cau truc notebook

| Phan | Noi dung |
|------|----------|
| 0 | Kiem tra moi truong |
| 1 | Import thu vien |
| 2A-2F | Constants, paths, embedding wrappers |
| 3A-3B | Product data pipeline |
| 4-4.4 | Qdrant, retriever, image retrieval |
| 5A-6 | Layer B outfit knowledge va category mapping |
| 7A-7B | Vision module |
| 8-10 | LLM, prompts, history, RAG chains |
| 11A-11D | Intent detection |
| 12 | Outfit context builder |
| 13A-13B | Security va chat loop chinh |


## PHẦN 0: Kiểm tra môi trường

Xác nhận Python executable đang dùng đúng virtual environment.

**Kết quả mong đợi:** đường dẫn chứa `/venv/` hoặc `conda`.

In [ ]:
import sys
print(f"Python executable: {sys.executable}")
print(f"Python version   : {sys.version}")
# Kết quả mong đợi: /venv/main/bin/python

## PHẦN 1: Import thư viện

Tất cả imports tập trung ở đây để dễ quản lý dependency.

**Phân nhóm:**
- `Standard library`: json, sys, uuid, os, time, base64, re
- `LangChain`: orchestration layer — embeddings, retrievers, chains, prompts, history
- `Qdrant`: vector database client
- `Ollama`: LLM + Vision-Language model
- `PyTorch / Transformers`: fine-tuned ViFashionCLIP
- `PIL`: image preprocessing

In [ ]:
# ── Standard library ──────────────────────────────────────────────────
import json, sys, uuid, os, time, base64, re

# ── PyTorch & Transformers (ViFashionCLIP) ────────────────────────────
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from transformers import AutoModel, AutoTokenizer, CLIPModel, CLIPProcessor

# ── LangChain — Embeddings & Documents ───────────────────────────────
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.embeddings import Embeddings
from langchain_core.documents import Document
from langchain_core.retrievers import BaseRetriever
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory

# ── LangChain — Chains ────────────────────────────────────────────────
from langchain_classic.chains import create_retrieval_chain, create_history_aware_retriever
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

# ── LangChain — Qdrant vector store ──────────────────────────────────
from langchain_qdrant import QdrantVectorStore

# ── Qdrant client ────────────────────────────────────────────────────
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams, Filter, FieldCondition, MatchAny
from qdrant_client.models import PointStruct

# ── Ollama ───────────────────────────────────────────────────────────
import ollama
from ollama import Client
from langchain_ollama import ChatOllama, OllamaEmbeddings

# ── Chat history ─────────────────────────────────────────────────────
from langchain_community.chat_message_histories import RedisChatMessageHistory

# ── Image processing ──────────────────────────────────────────────────
from PIL import Image
import io

# ── Utilities ────────────────────────────────────────────────────────
from tqdm import tqdm
from typing import Any

print("[OK] Import hoàn tất!")

## PHẦN 2A: Constants & Model Configuration

Tập trung tên model và hyperparameter ở một chỗ — khi đổi model chỉ sửa ở đây.

| Constant | Giá trị | Mục đích |
|----------|---------|----------|
| `LLM_MODEL` | `qwen3:4b-instruct` | Chat, Intent, Summarize |
| `QWEN_VL_MODEL` | `qwen2.5vl:3b` | Phân tích ảnh |
| `TEACHER_MODEL_NAME` | `patrickjohncyh/fashion-clip` | FashionCLIP gốc |
| `STUDENT_MODEL_NAME` | `AITeamVN/Vietnamese_Embedding_v2` | Encoder tiếng Việt |

In [ ]:
# ════════════════════════════════════════════════════════════════
# LLM & Vision-Language models (chạy local qua Ollama)
# ════════════════════════════════════════════════════════════════
LLM_MODEL     = "qwen3:4b-instruct"   # Chat chính + intent + summarize
QWEN_VL_MODEL = "qwen2.5vl:3b"        # Vision-Language: phân tích ảnh

# ════════════════════════════════════════════════════════════════
# ViFashionCLIP model names (Knowledge Distillation)
#   Teacher: FashionCLIP gốc → tạo ra soft labels 512-dim
#   Student: Vietnamese Embedding → học ánh xạ sang 512-dim
# ════════════════════════════════════════════════════════════════
TEACHER_MODEL_NAME   = "patrickjohncyh/fashion-clip"
STUDENT_MODEL_NAME   = "AITeamVN/Vietnamese_Embedding_v2"
STUDENT_MAX_LENGTH   = 128    # Max token length khi tokenize tiếng Việt

# ════════════════════════════════════════════════════════════════
# ProjectionHead hyperparameters
#   Ánh xạ: student_hidden_dim → FashionCLIP 512-dim
# ════════════════════════════════════════════════════════════════
PROJECTION_HIDDEN_DIM = 1024
PROJECTION_NUM_LAYERS = 3
PROJECTION_DROPOUT    = 0.05

print(f"[OK] Model config:")
print(f"     LLM_MODEL     : {LLM_MODEL}")
print(f"     QWEN_VL_MODEL : {QWEN_VL_MODEL}")
print(f"     TEACHER_MODEL : {TEACHER_MODEL_NAME}")
print(f"     STUDENT_MODEL : {STUDENT_MODEL_NAME}")

## PHẦN 2B: Tìm thư mục dự án & Checkpoint

`resolve_chatbot_fashion_dir()` tự động tìm `Chatbot_Fashion/` từ bất kỳ vị trí nào.

**Hỗ trợ 3 trường hợp:**
1. Chạy từ `notebooks/` → lên 1 cấp
2. Chạy từ `Chatbot_Fashion/` → trả về ngay
3. Chạy từ workspace root → tìm trong thư mục con

**Kết quả:** `CHATBOT_FASHION_DIR` và `VIFASHIONCLIP_CHECKPOINT` dùng cho toàn notebook.

In [ ]:
def resolve_chatbot_fashion_dir() -> Path:
    """
    Tìm thư mục gốc Chatbot_Fashion/ từ vị trí hiện tại.
    Hỗ trợ: chạy từ notebooks/, từ Chatbot_Fashion/, hoặc workspace root.
    """
    cwd = Path.cwd()
    if cwd.name == "notebooks" and cwd.parent.name == "Chatbot_Fashion":
        return cwd.parent
    if cwd.name == "Chatbot_Fashion":
        return cwd
    candidate = cwd / "Chatbot_Fashion"
    if candidate.exists():
        return candidate
    for parent in cwd.parents:
        candidate = parent / "Chatbot_Fashion"
        if candidate.exists():
            return candidate
    raise FileNotFoundError("Không tìm thấy Chatbot_Fashion/ từ thư mục hiện tại")


CHATBOT_FASHION_DIR      = resolve_chatbot_fashion_dir()
VIFASHIONCLIP_CHECKPOINT = CHATBOT_FASHION_DIR / "best_stage2_model.pt"

print(f"[OK] Thư mục dự án   : {CHATBOT_FASHION_DIR}")
print(f"[OK] Checkpoint path : {VIFASHIONCLIP_CHECKPOINT}")
print(f"     Checkpoint exists: {VIFASHIONCLIP_CHECKPOINT.exists()}")

## PHẦN 2C: Neural Network Blocks cho ProjectionHead

Các block tạo nên `ProjectionHead` — ánh xạ embedding tiếng Việt sang không gian FashionCLIP.

```
Text (VN) → StudentEncoder [hidden=768] → ProjectionHead → [512-dim FashionCLIP]
```

- `ResidualMLPBlock`: MLP + skip connection (tránh vanishing gradient)
- `ProjectionHead`: xếp ResidualMLPBlock thành mạng hoàn chỉnh
- `mean_pool`: pooling token embeddings → sentence embedding
- `Stage2StudentProjection`: ghép Encoder + Projection thành 1 forward pass

### Kiến trúc mạng cho ViFashionCLIP Student model

Gồm 4 khối, dùng lại lẫn nhau theo thứ tự:
1. `ResidualMLPBlock` — khối MLP con có skip-connection (dùng bên trong ProjectionHead)
2. `ProjectionHead` — ánh xạ embedding của student encoder sang không gian 512-dim của FashionCLIP
3. `mean_pool` — hàm pooling token embeddings thành 1 vector câu
4. `Stage2StudentProjection` — model hoàn chỉnh (encoder + ProjectionHead), dùng để load checkpoint khi inference

**`ResidualMLPBlock`** — MLP block có skip-connection, chống vanishing gradient.

In [ ]:
class ResidualMLPBlock(nn.Module):
    """
    MLP Block với skip connection: output = input + MLP(LayerNorm(input))
    Giúp gradient flow tốt hơn, tránh vanishing gradient trong mạng sâu.
    """
    def __init__(self, dim, dropout=0.05):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 2),   # Expand
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim),   # Contract
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.block(x)       # Residual

**`ProjectionHead`** — Ánh xạ student embedding → không gian 512-dim của FashionCLIP.

In [ ]:
class ProjectionHead(nn.Module):
    """
    Projection Head: ánh xạ student embedding → FashionCLIP teacher space.
    Kiến trúc: Linear → LayerNorm → GELU → [N×ResidualMLPBlock] → Linear(→512)
    """
    def __init__(self, input_dim, output_dim, hidden_dim=1024, num_layers=3, dropout=0.05):
        super().__init__()
        layers = [
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        ]
        for _ in range(max(0, num_layers - 2)):
            layers.append(ResidualMLPBlock(hidden_dim, dropout=dropout))
        layers.extend([
            nn.LayerNorm(hidden_dim),
            nn.Linear(hidden_dim, output_dim),  # Output: 512-dim
        ])
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

**`mean_pool`** — Pooling token embeddings thành 1 vector câu, bỏ qua padding.

In [ ]:
def mean_pool(last_hidden_state, attention_mask):
    """
    Mean pooling: trung bình có trọng số của token embeddings.
    Bỏ qua padding tokens (attention_mask=0), tránh chia 0.
    """
    mask   = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    denom  = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / denom

**`Stage2StudentProjection`** — Model tổng hợp Encoder + ProjectionHead, dùng khi inference/load checkpoint.

In [ ]:
class Stage2StudentProjection(nn.Module):
    """
    Model tổng hợp: Student Encoder + Projection Head.
    Forward: token_ids → mean_pool → projection → 512-dim vector.
    Đây là model được load từ checkpoint khi inference.
    """
    def __init__(self, encoder, projection_head):
        super().__init__()
        self.encoder         = encoder
        self.projection_head = projection_head

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled  = mean_pool(outputs.last_hidden_state, attention_mask)
        return self.projection_head(pooled)

print("[OK] Neural blocks ready: ResidualMLPBlock | ProjectionHead | mean_pool | Stage2StudentProjection")

## PHẦN 2D: BGE-M3 Embedding Wrapper (Layer B)

**Dùng cho:** Layer B — quy tắc phối đồ, phong cách, bối cảnh, dáng người, tone da
**KHÔNG dùng cho:** Layer A sản phẩm
**Vector:** 1024 chiều

> Prerequisite: `ollama pull bge-m3` và Ollama server tại `localhost:11434`

In [ ]:
class BGEM3Embeddings(Embeddings):
    """
    LangChain wrapper cho BGE-M3 qua Ollama.
    Dùng cho Layer B: phong cách, bối cảnh, dáng người, tone da, lý do tư vấn.
    Vector 1024 chiều. KHÔNG dùng cho Layer A.
    """

    def __init__(self, model_name: str = "bge-m3"):
        self.ollama_embeddings = OllamaEmbeddings(
            model    = model_name,
            base_url = "http://localhost:11434"
        )

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        """Embed nhiều documents — dùng khi index Layer B vào Qdrant."""
        return self.ollama_embeddings.embed_documents(texts)

    def embed_query(self, text: str) -> list[float]:
        """Embed 1 query — dùng khi tìm rule phối đồ cho user."""
        return self.ollama_embeddings.embed_query(text)


print("[OK] BGEM3Embeddings sẵn sàng (Layer B — 1024-dim)")

## PHẦN 2E: ViFashionCLIP Text Embedding (Layer A)

**Pipeline load model:**
```
checkpoint.pt → encoder_state_dict + projection_state_dict
    ↓                    ↓
AutoModel          ProjectionHead
    └──────────────────────┘
         Stage2StudentProjection (eval mode, no grad)
```

**Dùng cho:** Layer A — embed sản phẩm và search query tiếng Việt
**Vector:** 512 chiều — khớp với FashionCLIP teacher space

In [ ]:
class ViFashionCLIPTextEmbeddings(Embeddings):
    """
    LangChain Embeddings wrapper cho ViFashionCLIP tiếng Việt đã fine-tune.
    Dùng cho Layer A (embed sản phẩm + search query). Vector: 512 chiều.
    KHÔNG dùng cho Layer B (Layer B dùng BGE-M3).
    """

    def __init__(self,
                 checkpoint_path: str | Path = VIFASHIONCLIP_CHECKPOINT,
                 batch_size: int = 32):
        self.device           = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size       = batch_size
        self.checkpoint_path  = Path(checkpoint_path)

        if not self.checkpoint_path.exists():
            raise FileNotFoundError(f"Checkpoint không tồn tại: {self.checkpoint_path}")

        # ── Load checkpoint ────────────────────────────────────────────
        checkpoint   = torch.load(self.checkpoint_path, map_location="cpu")
        student_name = checkpoint.get("student_model_name", STUDENT_MODEL_NAME)
        teacher_dim  = int(checkpoint.get("teacher_dim", 512))

        # ── Khởi tạo tokenizer + encoder ──────────────────────────────
        self.tokenizer = AutoTokenizer.from_pretrained(student_name)
        encoder        = AutoModel.from_pretrained(student_name)

        # ── Khởi tạo ProjectionHead ────────────────────────────────────
        projection = ProjectionHead(
            input_dim  = encoder.config.hidden_size,
            output_dim = teacher_dim,
            hidden_dim = PROJECTION_HIDDEN_DIM,
            num_layers = PROJECTION_NUM_LAYERS,
            dropout    = PROJECTION_DROPOUT,
        )

        # ── Load weights & chuyển sang eval mode ──────────────────────
        encoder.load_state_dict(checkpoint["encoder_state_dict"])
        projection.load_state_dict(checkpoint["projection_state_dict"])
        self.model = Stage2StudentProjection(encoder, projection).to(self.device).eval()
        for p in self.model.parameters():
            p.requires_grad = False  # Freeze — chỉ inference

        self.embedding_dim = teacher_dim
        print(f"[OK] ViFashionCLIP loaded: {self.checkpoint_path}")
        print(f"     Device: {self.device} | Dim: {self.embedding_dim}")

    @torch.no_grad()
    def _encode(self, texts: list[str]) -> list[list[float]]:
        """
        Encode batch texts → 512-dim L2-normalized vectors.
        Tự chia thành mini-batches. Dùng tqdm progress bar khi index lớn.
        """
        all_embeds = []
        for i in tqdm(range(0, len(texts), self.batch_size),
                      desc="ViFashionCLIP embed", leave=False):
            batch  = [str(x) for x in texts[i:i + self.batch_size]]
            inputs = self.tokenizer(
                batch, padding=True, truncation=True,
                max_length=STUDENT_MAX_LENGTH, return_tensors="pt",
            ).to(self.device)
            embeds = self.model(
                input_ids      = inputs["input_ids"],
                attention_mask = inputs["attention_mask"]
            )
            embeds = F.normalize(embeds, p=2, dim=-1)  # L2 normalize
            all_embeds.append(embeds.detach().cpu())
        return torch.cat(all_embeds, dim=0).tolist()

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        """Embed nhiều documents — dùng khi index sản phẩm vào Qdrant."""
        return self._encode(texts)

    def embed_query(self, text: str) -> list[float]:
        """Embed 1 query — dùng khi user tìm kiếm sản phẩm."""
        return self._encode([text])[0]


print("[OK] ViFashionCLIPTextEmbeddings sẵn sàng (Layer A text — 512-dim)")

## PHẦN 2F: FashionCLIP Image Embedding

Image encoder dùng FashionCLIP gốc (teacher model) để encode ảnh sản phẩm.

**Dùng cho:**
- Index ảnh MAIN sản phẩm vào Qdrant (offline, 1 lần)
- Encode ảnh user gửi để tìm sản phẩm tương tự (online)

**Lazy-load:** class chỉ khởi tạo khi thực sự cần (xem PHẦN 4.2),
tránh tốn RAM/GPU khi chỉ chat text.

In [ ]:
class FashionCLIPImageEmbeddings:
    """
    FashionCLIP image encoder cho image retrieval.
    Dùng cho: index ảnh MAIN → 512-dim | encode ảnh query → tìm sản phẩm tương tự.
    Layer B vẫn dùng BGE-M3. Product text search vẫn dùng ViFashionCLIPTextEmbeddings.
    """

    def __init__(self, model_name: str = TEACHER_MODEL_NAME, batch_size: int = 32):
        self.device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.batch_size = batch_size
        self.model_name = model_name

        self.processor = CLIPProcessor.from_pretrained(model_name)
        self.model     = CLIPModel.from_pretrained(model_name).to(self.device).eval()
        for p in self.model.parameters():
            p.requires_grad = False  # Chỉ inference

        self.embedding_dim = int(getattr(self.model.config, "projection_dim", 512))
        print(f"[OK] FashionCLIP image encoder: {model_name}")
        print(f"     Device: {self.device} | Dim: {self.embedding_dim}")

    @staticmethod
    def _open_image(image_path: str | Path):
        """Mở ảnh và convert sang RGB (xử lý PNG có alpha channel)."""
        return Image.open(image_path).convert("RGB")

    @torch.no_grad()
    def encode_image_paths(self, image_paths: list[str | Path]) -> list[list[float] | None]:
        """
        Encode nhiều ảnh → 512-dim vectors.
        Ảnh lỗi (không đọc được) → None, indexing bỏ qua.
        """
        results: list[list[float] | None] = [None] * len(image_paths)

        for start in tqdm(range(0, len(image_paths), self.batch_size),
                          desc="FashionCLIP image embed", leave=False):
            batch_paths     = image_paths[start:start + self.batch_size]
            images          = []
            valid_positions = []

            for offset, path in enumerate(batch_paths):
                try:
                    images.append(self._open_image(path))
                    valid_positions.append(start + offset)
                except Exception as e:
                    print(f"[WARN] Không đọc ảnh {path}: {e}")

            if not images:
                continue

            inputs = self.processor(
                images=images, return_tensors="pt", padding=True
            ).to(self.device)
            embeds = self.model.get_image_features(**inputs)
            if hasattr(embeds, "pooler_output"):
                embeds = embeds.pooler_output
            embeds = F.normalize(embeds, p=2, dim=-1).detach().cpu().tolist()

            for pos, vec in zip(valid_positions, embeds):
                results[pos] = vec

        return results

    def embed_image(self, image_path: str | Path) -> list[float] | None:
        """Encode 1 ảnh — shortcut cho image search query."""
        return self.encode_image_paths([image_path])[0]


print("[OK] FashionCLIPImageEmbeddings sẵn sàng (Image Retrieval — 512-dim)")
print("     Lazy-load: chỉ khởi tạo khi cần (xem get_image_embeddings() ở PHẦN 4.2)")

## PHẦN 3A: Data Pipeline — Tiền xử lý metadata sản phẩm

Chuyển đổi từng sản phẩm từ JSON thô thành `LangChain Document`.

```
JSONL file (65k dòng)
    ↓  json.loads()
raw product dict
    ├── extract_image_urls()         → lấy URLs ảnh
    ├── build_product_metadata()     → dict cho filter & display
    └── build_product_page_content() → text để embed (có nhãn rõ)
    ↓
LangChain Document(page_content, metadata)
```

**Tại sao dùng format có nhãn?** ViFashionCLIP fine-tune với text dạng
`"Tên sản phẩm: ... Màu sắc: ..."` — format này giúp embedding chính xác hơn.

### Tiện ích xử lý metadata sản phẩm (dùng khi build Document cho Layer A)

- `extract_image_urls` — lấy URL ảnh chất lượng cao từ metadata thô
- `normalize_to_text` — chuẩn hoá 1 field bất kỳ (None/list/str) thành chuỗi hiển thị
- `build_product_metadata` — build metadata dict cho LangChain Document (dùng để filter/hiển thị)
- `build_product_page_content` — build đoạn text đầy đủ để embed (ViFashionCLIP)

**`extract_image_urls`** — Lấy toàn bộ URL ảnh 'large' của sản phẩm.

In [ ]:
def extract_image_urls(item: dict) -> list[str]:
    """
    Lấy tất cả URL ảnh (large) từ field "images" của sản phẩm.
    Chỉ lấy ảnh có field "large" (chất lượng cao nhất).
    """
    return [img.get("large") for img in item.get("images", []) if img.get("large")]

**`normalize_to_text`** — Chuẩn hoá 1 field (None/list/str) → chuỗi hiển thị, có giá trị mặc định.

In [ ]:
def normalize_to_text(value, default: str = "Không rõ") -> str:
    """
    Chuẩn hóa field thành chuỗi cho page_content.
    Xử lý: None/empty → default, list → join bằng dấu phẩy, value → str.
    """
    if value is None or value == "":
        return default
    if isinstance(value, list):
        cleaned = [str(x).strip() for x in value if str(x).strip()]
        return ", ".join(cleaned) if cleaned else default
    return str(value).strip() or default

**`build_product_metadata`** — Build metadata dict cho Document (product_id, category, ảnh, giá...).

In [ ]:
def build_product_metadata(item: dict) -> dict:
    """
    Tạo metadata dict cho LangChain Document.
    Dùng để: filter Qdrant, display kết quả, render ảnh sản phẩm.
    """
    image_urls = extract_image_urls(item)
    return {
        "product_id" : item.get("product_id", ""),
        "title"      : item.get("title", ""),
        "category"   : item.get("category", ""),
        "department" : item.get("department", ""),
        "brand"      : item.get("brand", ""),
        "price"      : item.get("price", 0),
        "images"     : image_urls,
        "image_url"  : image_urls[0] if image_urls else "",
    }

**`build_product_page_content`** — Build đoạn text có nhãn rõ ràng để embed bằng ViFashionCLIP.

In [ ]:
def build_product_page_content(item: dict) -> str:
    """
    Tạo text để embed bằng ViFashionCLIP.
    Format có nhãn rõ ràng giúp model nhận biết từng field:
      "Tên sản phẩm: Áo thun trắng\nMàu sắc: Trắng\nDịp: Đi học"
    """
    details = item.get("details", {}) or {}
    fields  = [
        ("Tên sản phẩm", item.get("title")),
        ("Mã sản phẩm" , item.get("product_id")),
        ("Danh mục"    , item.get("category")),
        ("Đối tượng"   , item.get("department")),
        ("Thương hiệu" , item.get("brand")),
        ("Giá"         , f"{item.get('price', 0)} VND"),
        ("Màu sắc"     , details.get("main_color")),
        ("Chất liệu"   , details.get("material")),
        ("Kích cỡ"     , details.get("size")),
        ("Họa tiết"    , details.get("pattern")),
        ("Mùa phù hợp" , item.get("season")),
        ("Dịp sử dụng" , item.get("occasion")),
        ("Mô tả"       , item.get("description")),
    ]
    return "\n".join(f"{label}: {normalize_to_text(value)}" for label, value in fields)

print("[OK] Metadata utilities ready: extract_image_urls | normalize_to_text | build_product_metadata | build_product_page_content")

## PHẦN 3B: Data Pipeline — Đọc JSONL & Index vào Qdrant

**`process_fashion_metadata()`** — đọc file JSONL → list of Documents
**`run_data_pipeline()`** — end-to-end: đọc → embed → upsert Qdrant

**Cơ chế Resume:** dựa vào số point trong collection.
Nếu đã có 30k points → bỏ qua 30k đầu, index tiếp từ sản phẩm 30001.

> ⚠️ **Chạy 1 lần duy nhất** khi khởi tạo hoặc thêm dữ liệu mới.
> Hàm tự kiểm tra và bỏ qua nếu đã đủ.

### Đọc file JSONL sản phẩm & index vào Qdrant (Layer A — text)

- `process_fashion_metadata` — đọc từng dòng JSONL → list Document, kèm thống kê lỗi
- `run_data_pipeline` — pipeline end-to-end: đọc file → embed ViFashionCLIP → upsert Qdrant (resume tự động)

**`process_fashion_metadata`** — Đọc JSONL → list[Document], thống kê dòng lỗi/thiếu field.

In [ ]:
def process_fashion_metadata(file_path: str | Path) -> list:
    """
    Đọc file JSONL → list of LangChain Documents.
    Thu thập thống kê: lỗi JSON, thiếu ID/category/ảnh.
    """
    file_path = Path(file_path)
    documents = []
    stats     = {
        "total_lines"       : 0,
        "json_errors"       : 0,
        "missing_product_id": 0,
        "missing_category"  : 0,
        "missing_image"     : 0,
    }

    with file_path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            stats["total_lines"] += 1
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                stats["json_errors"] += 1
                continue

            metadata = build_product_metadata(item)
            if not metadata["product_id"]: stats["missing_product_id"] += 1
            if not metadata["category"]:   stats["missing_category"]   += 1
            if not metadata["image_url"]:  stats["missing_image"]      += 1

            documents.append(Document(
                page_content = build_product_page_content(item),
                metadata     = metadata,
            ))

    print(
        f"[THỐNG KÊ] dòng={stats['total_lines']} | docs={len(documents)} | "
        f"lỗi_json={stats['json_errors']} | "
        f"thiếu_id={stats['missing_product_id']} | "
        f"thiếu_category={stats['missing_category']} | "
        f"thiếu_ảnh={stats['missing_image']}"
    )
    return documents

**`run_data_pipeline`** — Pipeline index Layer A: đọc → embed → upsert Qdrant, resume theo points_count.

In [ ]:
def run_data_pipeline(
    metadata_file  : str | Path = CHATBOT_FASHION_DIR / "data" / "metadata" / "meta_Amazon_Lazada_Fashion_65k.jsonl",
    qdrant_url     : str        = "http://localhost:6333",
    collection_name: str        = "fashion_products_vifashionclip_vi_65k_structured_vi",
):
    """
    End-to-end indexing: đọc JSONL → embed ViFashionCLIP → upsert Qdrant.
    Resume tự động dựa trên số point hiện có (giữ thứ tự file ổn định).
    """
    metadata_file = Path(metadata_file)
    if not metadata_file.exists():
        print(f"[LỖI] Không tìm thấy: {metadata_file}")
        return

    print(f"[THÔNG BÁO] Đọc dữ liệu từ: {metadata_file}")
    all_docs = process_fashion_metadata(metadata_file)

    emb    = ViFashionCLIPTextEmbeddings()
    client = QdrantClient(url=qdrant_url)

    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name = collection_name,
            vectors_config  = VectorParams(size=512, distance=Distance.COSINE),
        )
        current_count = 0
    else:
        current_count = client.get_collection(collection_name).points_count
        print(f"[INFO] Collection đã có {current_count} sản phẩm.")

    remaining = all_docs[current_count:]
    if not remaining:
        print("[OK] Đã index đầy đủ!")
        return

    vdb = QdrantVectorStore(client=client, collection_name=collection_name, embedding=emb)
    with tqdm(total=len(all_docs), initial=current_count, desc="Index", unit="SP") as pbar:
        for i in range(0, len(remaining), 128):
            vdb.add_documents(documents=remaining[i:i+128])
            pbar.update(min(128, len(remaining) - i))

    print("[OK] Index hoàn tất!")

# ⚠️ Bỏ comment NẾU muốn index lần đầu (tốn nhiều thời gian)
# run_data_pipeline()

print("[OK] Data Pipeline sẵn sàng. Gọi run_data_pipeline() để index lần đầu.")

## PHẦN 4: Kết nối Qdrant & Khởi tạo Embedding Objects

Khởi tạo `product_embeddings` (Layer A) và `rule_embeddings` (Layer B),
kết nối Qdrant.

**Hai chế độ kết nối:**
- `Option A (local path)`: không cần Docker ✅ đang dùng
- `Option B (server)`: kết nối Qdrant qua HTTP

> **Không dùng lẫn lộn** `product_embeddings` và `rule_embeddings` — chúng
> dùng cho hai collection khác nhau với vector dim khác nhau.

In [ ]:
print("[INFO] Khởi tạo embedding models...")

# Layer A: ViFashionCLIP tiếng Việt — 512-dim
product_embeddings = ViFashionCLIPTextEmbeddings()

# Layer B: BGE-M3 qua Ollama — 1024-dim
# Lưu ý: rule_embeddings KHÔNG dùng cho product retrieval
rule_embeddings = BGEM3Embeddings()

print("[INFO] Kết nối Qdrant...")

# Option A: Qdrant local folder (không cần Docker)
QDRANT_LOCAL_PATH = CHATBOT_FASHION_DIR / "qdrant_data_65k" / "qdrant_data"
client = QdrantClient(path=str(QDRANT_LOCAL_PATH))

# Option B: Qdrant server / Docker
# client = QdrantClient(url="http://localhost:6333")

print(f"[OK] Qdrant kết nối tại: {QDRANT_LOCAL_PATH}")

## PHẦN 4.1: Product Collection & Diversity Retriever

Cấu hình retriever cho Layer A (sản phẩm).

**Chiến lược retrieval 2 bước:**
```
Query → ViFashionCLIP embed → Qdrant top-30
                    ↓
    DiversityFilteredRetriever
    ├── Loại duplicate (cùng product_id)
    ├── Giới hạn brand (max 2/brand)
    └── Trả về top-5 đa dạng
```

**Không limit theo category** để hỗ trợ query đơn category
(VD: 'áo thun trắng' → user muốn nhiều áo thun).

### Product Collection setup — kết nối Qdrant collection sản phẩm (Layer A text)

In [ ]:
PRODUCT_COLLECTION         = "fashion_products_vifashionclip_vi_65k_structured_vi"
PRODUCT_SEARCH_CANDIDATE_K = 30   # Ứng viên từ Qdrant trước khi lọc
PRODUCT_SEARCH_PAGE_SIZE   = 5    # Sản phẩm tối đa gửi vào LLM
PRODUCT_SEARCH_BRAND_LIMIT = 2    # Tối đa 2 sản phẩm cùng brand

vector_db = QdrantVectorStore(
    client          = client,
    collection_name = PRODUCT_COLLECTION,
    embedding       = product_embeddings,
)

### Diversity-filtered retriever — tránh trả về nhiều sản phẩm trùng brand/sản phẩm

- `normalize_product_metadata` — đảm bảo mỗi Document có `image_url` ổn định
- `diversity_filter_documents` — lọc bớt trùng product_id, giới hạn tối đa N sản phẩm/brand
- `DiversityFilteredRetriever` — Retriever wrapper: gọi Qdrant rồi áp dụng diversity filter

In [ ]:
def normalize_product_metadata(doc: Document) -> Document:
    """
    Đảm bảo mỗi Document sản phẩm có image_url ổn định để render.
    Xử lý cả trường hợp images là string thay vì list.
    """
    images = doc.metadata.get("images", [])
    if isinstance(images, str):
        images = [images] if images else []
    doc.metadata["image_url"] = doc.metadata.get("image_url") or (images[0] if images else "")
    return doc

In [ ]:
def diversity_filter_documents(
    docs        : list[Document],
    max_docs    : int = PRODUCT_SEARCH_PAGE_SIZE,
    max_per_brand: int = PRODUCT_SEARCH_BRAND_LIMIT,
) -> list[Document]:
    """
    Lọc kết quả Qdrant: loại duplicate product_id, giới hạn brand.
    2 bước:
      Pass 1 — brand diversity: max 2 SP/brand
      Pass 2 — fill slot còn thiếu bằng SP chưa thấy (bỏ qua giới hạn brand)
    """
    selected         : list[Document] = []
    seen_product_ids : set[str]       = set()
    brand_counts     : dict[str, int] = {}

    # Pass 1: brand diversity
    for doc in docs:
        doc        = normalize_product_metadata(doc)
        product_id = str(doc.metadata.get("product_id", "")).strip().lower()
        brand      = str(doc.metadata.get("brand", "")).strip().lower()

        if product_id and product_id in seen_product_ids:
            continue
        if brand and brand_counts.get(brand, 0) >= max_per_brand:
            continue

        selected.append(doc)
        if product_id: seen_product_ids.add(product_id)
        if brand:      brand_counts[brand] = brand_counts.get(brand, 0) + 1
        if len(selected) >= max_docs:
            return selected

    # Pass 2: fill remaining
    for doc in docs:
        if len(selected) >= max_docs:
            break
        doc        = normalize_product_metadata(doc)
        product_id = str(doc.metadata.get("product_id", "")).strip().lower()
        if product_id and product_id in seen_product_ids:
            continue
        selected.append(doc)
        if product_id: seen_product_ids.add(product_id)

    return selected

### Khởi tạo retriever chính thức dùng trong RAG chain

In [ ]:
class DiversityFilteredRetriever(BaseRetriever):
    """Retriever wrapper: fetch candidates từ Qdrant, rồi dedupe/diversify."""
    base_retriever: Any
    max_docs: int = PRODUCT_SEARCH_PAGE_SIZE

    def _get_relevant_documents(self, query: str, *, run_manager=None) -> list[Document]:
        raw_docs = self.base_retriever.invoke(query)
        return diversity_filter_documents(raw_docs, max_docs=self.max_docs)


base_retriever = vector_db.as_retriever(
    search_type   = "similarity",
    search_kwargs = {"k": PRODUCT_SEARCH_CANDIDATE_K},
)
retriever = DiversityFilteredRetriever(
    base_retriever = base_retriever,
    max_docs       = PRODUCT_SEARCH_PAGE_SIZE,
)

print(f"[OK] Product collection : {PRODUCT_COLLECTION} (512-dim)")
print(f"[OK] Retriever          : top-{PRODUCT_SEARCH_CANDIDATE_K} → page {PRODUCT_SEARCH_PAGE_SIZE}")

## PHẦN 4.2: Image Retrieval — Constants & Lazy Loader

Constants cho image collection và lazy loader cho `FashionCLIPImageEmbeddings`.

**Tại sao lazy-load?**
- `FashionCLIPImageEmbeddings` tải `CLIPModel` (~450MB) lên GPU/CPU
- Khi chỉ dùng text chat, không cần model này
- Chỉ gọi `get_image_embeddings()` khi user gửi ảnh sản phẩm

In [ ]:
# Constants cho Image Retrieval
PRODUCT_IMAGE_COLLECTION = "fashion_products_fashionclip_image_main_65k"
PRODUCT_IMAGE_ROOT       = CHATBOT_FASHION_DIR / "data" / "images"

# Lazy-load: tránh load model lớn khi không cần
_image_embeddings_instance = None


def get_image_embeddings() -> FashionCLIPImageEmbeddings:
    """
    Singleton lazy-loader cho FashionCLIPImageEmbeddings.
    Gọi lần đầu → load model (~30s). Các lần sau → trả về instance cũ.
    """
    global _image_embeddings_instance
    if _image_embeddings_instance is None:
        print("[INFO] Load FashionCLIP image encoder lần đầu...")
        _image_embeddings_instance = FashionCLIPImageEmbeddings(batch_size=32)
    return _image_embeddings_instance


print(f"[OK] Image Retrieval constants:")
print(f"     Collection : {PRODUCT_IMAGE_COLLECTION}")
print(f"     Image root : {PRODUCT_IMAGE_ROOT}")
print(f"     Lazy-load  : gọi get_image_embeddings() khi cần")

## PHẦN 4.3: Image Retrieval — Resolve đường dẫn ảnh MAIN

3 helper functions để:
1. Tìm ảnh MAIN trong metadata sản phẩm (fallback qua variant → URL → product_id)
2. Chuyển relative path từ metadata sang absolute local path
3. Generator: yield từng sản phẩm có ảnh MAIN tồn tại trên disk

### Resolve đường dẫn ảnh MAIN của sản phẩm trên đĩa

- `get_main_image_relative_path` — lấy relative path ảnh MAIN từ metadata (3 bước fallback)
- `resolve_main_image_path` — chuyển relative path → absolute path trên đĩa
- `iter_products_with_main_image` — generator: chỉ yield sản phẩm có ảnh MAIN tồn tại thật

**`get_main_image_relative_path`** — Lấy relative path ảnh MAIN, fallback 3 bước.

In [ ]:
def get_main_image_relative_path(item: dict) -> str:
    """
    Lấy relative path ảnh MAIN từ metadata sản phẩm.
    Fallback theo 3 bước:
      1. Ảnh có variant=="MAIN" và field "large"
      2. Ảnh có "_MAIN" trong tên URL
      3. Ảnh đầu tiên trong list (nếu có)
    """
    images = item.get("images", []) or []

    for img in images:
        if str(img.get("variant", "")).upper() == "MAIN" and img.get("large"):
            return img["large"]

    for img in images:
        large = str(img.get("large", ""))
        if "_MAIN" in large.upper():
            return large

    if images and images[0].get("large"):
        return images[0]["large"]

    product_id = item.get("product_id")
    return f"images/{product_id}_MAIN.jpg" if product_id else ""

**`resolve_main_image_path`** — Chuyển relative path metadata → absolute path trên đĩa.

In [ ]:
def resolve_main_image_path(relative_path: str,
                            image_root: str | Path = PRODUCT_IMAGE_ROOT) -> Path:
    """
    Chuyển metadata path (vd: "images/ABC_MAIN.jpg") → local absolute path.
    Bỏ prefix "images/" nếu có.
    """
    rel = Path(str(relative_path).replace("\\", "/"))
    if rel.parts and rel.parts[0].lower() == "images":
        rel = Path(*rel.parts[1:])
    return Path(image_root) / rel

**`iter_products_with_main_image`** — Generator yield (item, rel_path, img_path) cho sản phẩm có ảnh MAIN tồn tại.

In [ ]:
def iter_products_with_main_image(
    metadata_file: str | Path,
    image_root   : str | Path = PRODUCT_IMAGE_ROOT,
):
    """
    Generator: yield (item, rel_path, img_path) cho từng sản phẩm
    có ảnh MAIN tồn tại trên disk.
    Bỏ qua sản phẩm thiếu ảnh hoặc JSON lỗi.
    """
    metadata_file = Path(metadata_file)
    image_root    = Path(image_root)

    with metadata_file.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                item = json.loads(line)
            except json.JSONDecodeError:
                continue

            rel_path = get_main_image_relative_path(item)
            img_path = resolve_main_image_path(rel_path, image_root)
            if not img_path.exists():
                continue

            yield item, rel_path, img_path

print("[OK] Image path helpers ready: get_main_image_relative_path | resolve_main_image_path | iter_products_with_main_image")

## PHẦN 4.4: Image Retrieval — Index & Search

**`run_main_image_index_pipeline()`** — index 1 ảnh MAIN/sản phẩm vào Qdrant (chạy 1 lần).
**`image_point_to_document()`** — convert Qdrant point → LangChain Document.
**`search_products_by_image()`** — tìm sản phẩm tương tự qua ảnh.

**Luồng index:**
```
JSONL → iter_products_with_main_image → encode_image_paths → PointStruct → client.upsert
```

**Luồng search:**
```
user_image → embed_image → client.query_points → image_point_to_document → Documents
```

### Index & tìm kiếm sản phẩm bằng ảnh (FashionCLIP image embedding)

- `build_image_payload` — build payload lưu kèm vector ảnh trong Qdrant
- `run_main_image_index_pipeline` — index 1 ảnh MAIN/sản phẩm vào Qdrant (chạy 1 lần, resume tự động)
- `image_point_to_document` — convert Qdrant point (ảnh) → LangChain Document
- `search_products_by_image` — tìm sản phẩm tương tự từ 1 ảnh query

**`build_image_payload`** — Build payload (kèm page_content) lưu cùng vector ảnh trong Qdrant.

In [ ]:
def build_image_payload(item: dict, rel_path: str, img_path: Path) -> dict:
    """
    Tạo payload lưu kèm vector ảnh trong Qdrant.
    Bao gồm cả page_content để LLM có thể trả lời chi tiết sản phẩm.
    """
    metadata = build_product_metadata(item)
    metadata["main_image_path"]   = str(img_path)
    metadata["main_image_relpath"] = rel_path
    metadata["image_url"]          = metadata.get("image_url") or rel_path

    return {
        "product_id"        : metadata.get("product_id", ""),
        "title"             : metadata.get("title", ""),
        "category"          : metadata.get("category", ""),
        "department"        : metadata.get("department", ""),
        "brand"             : metadata.get("brand", ""),
        "price"             : metadata.get("price", 0),
        "image_url"         : metadata.get("image_url", ""),
        "main_image_path"   : str(img_path),
        "main_image_relpath": rel_path,
        "page_content"      : build_product_page_content(item),
        "metadata"          : metadata,
    }

**`run_main_image_index_pipeline`** — Pipeline index ảnh MAIN → FashionCLIP 512-dim → Qdrant, resume tự động.

In [ ]:
def run_main_image_index_pipeline(
    metadata_file  : str | Path = CHATBOT_FASHION_DIR / "data" / "metadata" / "meta_Amazon_Lazada_Fashion_65k.jsonl",
    image_root     : str | Path = PRODUCT_IMAGE_ROOT,
    collection_name: str        = PRODUCT_IMAGE_COLLECTION,
    batch_size     : int        = 64,
):
    """
    Index 1 ảnh MAIN mỗi sản phẩm vào Qdrant bằng FashionCLIP image vector.
    Resume tự động: nếu collection đã có N vectors → bỏ qua N sản phẩm đầu.
    ⚠️ Chạy 1 lần, mất ~20-30 phút với 65k sản phẩm trên GPU.
    """
    metadata_file = Path(metadata_file)
    image_root    = Path(image_root)

    if not metadata_file.exists():
        print(f"[LỖI] Không tìm thấy metadata: {metadata_file}")
        return
    if not image_root.exists():
        print(f"[LỖI] Không tìm thấy thư mục ảnh: {image_root}")
        return

    print(f"[INFO] Đang scan ảnh MAIN tại: {image_root}")
    items = list(iter_products_with_main_image(metadata_file, image_root))
    print(f"[OK] Tìm thấy {len(items)} sản phẩm có ảnh MAIN")

    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name = collection_name,
            vectors_config  = VectorParams(size=512, distance=Distance.COSINE),
        )
        current_count = 0
    else:
        current_count = client.count(collection_name).count
        print(f"[INFO] Collection đã có {current_count} vectors")

    remaining = items[current_count:]
    if not remaining:
        print("[OK] Ảnh MAIN đã được index đầy đủ!")
        return

    emb = get_image_embeddings()
    print(f"[INFO] Đang index {len(remaining)} ảnh còn lại...")

    with tqdm(total=len(items), initial=current_count,
              desc="Image index", unit="img") as pbar:
        for start in range(0, len(remaining), batch_size):
            batch       = remaining[start:start + batch_size]
            image_paths = [img_path for _, _, img_path in batch]
            vectors     = emb.encode_image_paths(image_paths)

            points = []
            for (item, rel_path, img_path), vector in zip(batch, vectors):
                if vector is None:
                    continue
                product_id = str(item.get("product_id", ""))
                point_id   = str(uuid.uuid5(uuid.NAMESPACE_URL, f"fashion-main-image:{product_id}"))
                points.append(PointStruct(
                    id      = point_id,
                    vector  = vector,
                    payload = build_image_payload(item, rel_path, img_path),
                ))

            if points:
                client.upsert(collection_name=collection_name, points=points)
            pbar.update(len(batch))

    print(f"[OK] Index ảnh hoàn tất → {collection_name}")

**`image_point_to_document`** — Convert 1 Qdrant point (ảnh) → Document dùng chung với prompt LLM.

In [ ]:
def image_point_to_document(point) -> Document:
    """Convert Qdrant image point → LangChain Document để dùng chung với prompt."""
    payload  = point.payload or {}
    metadata = payload.get("metadata", {}) or {}
    metadata["image_search_score"] = getattr(point, "score", None)
    metadata["image_url"] = payload.get("image_url") or metadata.get("image_url", "")
    return Document(
        page_content = payload.get("page_content", ""),
        metadata     = metadata,
    )

**`search_products_by_image`** — Tìm tối đa N sản phẩm tương tự ảnh query, mỗi product_id lấy 1 kết quả.

In [ ]:
def search_products_by_image(
    image_path     : str | Path,
    collection_name: str        = PRODUCT_IMAGE_COLLECTION,
    top_k          : int        = 12,
    max_products   : int        = 3,
    score_threshold: float|None = 0.15,
) -> list[Document]:
    """
    Tìm sản phẩm tương tự qua ảnh.
    Trả về tối đa max_products Documents, mỗi product_id chỉ lấy 1 kết quả.
    """
    image_path = Path(image_path)
    if not image_path.exists():
        print(f"[LỖI] Không tìm thấy ảnh: {image_path}")
        return []
    if not client.collection_exists(collection_name):
        print(f"[WARN] Collection ảnh chưa tồn tại. Chạy run_main_image_index_pipeline() trước.")
        return []

    query_vector = get_image_embeddings().embed_image(image_path)
    if query_vector is None:
        return []

    kwargs = {
        "collection_name": collection_name,
        "query"          : query_vector,
        "limit"          : top_k,
        "with_payload"   : True,
    }
    if score_threshold is not None:
        kwargs["score_threshold"] = score_threshold

    response = client.query_points(**kwargs)

    docs, seen_ids = [], set()
    for point in response.points:
        doc        = image_point_to_document(point)
        product_id = doc.metadata.get("product_id", "")
        if product_id and product_id in seen_ids:
            continue
        docs.append(normalize_product_metadata(doc))
        if product_id: seen_ids.add(product_id)
        if len(docs) >= max_products:
            break

    return docs

# ⚠️ Bỏ comment dòng dưới NẾU muốn index ảnh lần đầu (~20-30 phút)
# run_main_image_index_pipeline()

print("[OK] Image Retrieval sẵn sàng: MAIN image → FashionCLIP 512-dim → Qdrant")
print(f"     Collection: {PRODUCT_IMAGE_COLLECTION}")

## PHẦN 5A: Layer B — Nạp dữ liệu quy tắc phối đồ

Nạp file JSON chứa quy tắc phối đồ (kiến thức stylist) cho cả Nữ và Nam.

**Cấu trúc mỗi rule:**
```json
{
  "rule_key": "Áo mặc trong (áo thun/sơ mi)|Áo thun basic",
  "phong_cach": "Casual",
  "boi_canh": "Đi chơi cuối tuần",
  "dang_nguoi": "Dáng chữ nhật",
  "tone_da": "Da trung bình",
  "ly_do_tu_van": "...",
  "goi_y_phoi_cung": ["Quần/Chân váy", "Giày dép"]
}
```

In [ ]:
def load_layer_b(file_path: str | Path) -> list:
    """Nạp file JSON chứa quy tắc phối đồ Layer B."""
    with open(file_path, "r", encoding="utf-8") as f:
        return json.load(f)


layer_b_female = load_layer_b(CHATBOT_FASHION_DIR / "data" / "stylists" / "Layer_B_Female_Knowledge.json")
layer_b_male   = load_layer_b(CHATBOT_FASHION_DIR / "data" / "stylists" / "Layer_B_Male_Knowledge.json")

print(f"[OK] Layer B: {len(layer_b_female)} rules Nữ | {len(layer_b_male)} rules Nam")

## PHẦN 5B: Layer B — Index vào Qdrant

Index rules vào hai collection riêng biệt: `layer_b_female` và `layer_b_male`.

**Text được embed:** `rule_key + phong_cach + boi_canh + ly_do_tu_van`
(ghép nhiều field có ngữ nghĩa → BGE-M3 xử lý tốt cả tiếng Việt + tiếng Anh)

> ⚠️ Chỉ chạy 1 lần — hàm tự kiểm tra và bỏ qua nếu collection đã tồn tại.

In [ ]:
def index_layer_b(data: list, collection_name: str):
    """
    Index toàn bộ Layer B vào Qdrant bằng BGE-M3 (chạy 1 lần).
    Nếu collection đã tồn tại → bỏ qua, không overwrite.
    """
    existing = [c.name for c in client.get_collections().collections]
    if collection_name in existing:
        count = client.count(collection_name).count
        print(f"[SKIP] {collection_name} đã tồn tại ({count} rules) — bỏ qua index.")
        return

    client.create_collection(
        collection_name = collection_name,
        vectors_config  = VectorParams(size=1024, distance=Distance.COSINE),
    )

    points = []
    for i, rule in enumerate(data):
        # Ghép các field có ngữ nghĩa để embed
        text   = (f"{rule['rule_key']} {rule['phong_cach']} "
                  f"{rule['boi_canh']} {rule['ly_do_tu_van']}")
        vector = rule_embeddings.embed_query(text)
        points.append(PointStruct(id=i, vector=vector, payload=rule))

    client.upsert(collection_name=collection_name, points=points)
    print(f"[OK] Indexed {len(points)} rules → {collection_name}")


index_layer_b(layer_b_female, "layer_b_female")
index_layer_b(layer_b_male,   "layer_b_male")

## PHẦN 5C: Layer B — Tìm rule & chi tiết phối đồ

**`find_matching_rule()`** — tìm quy tắc phối đồ phù hợp nhất.
Fallback 3 bước: lọc (Dáng + Da) → lọc (Dáng only) → không lọc.

**`find_outfit_details()`** — từ base rule, tìm chi tiết từng món đồ.
2 cách: khớp chính xác bằng text → fallback semantic search Qdrant.

### Tìm quy tắc phối đồ phù hợp & chi tiết từng món (Layer B)

- `find_matching_rule` — tìm công thức phối đồ chủ đạo (fallback 3 bước theo dáng/da)
- `find_outfit_details` — từ công thức chủ đạo, tìm chi tiết từng món đồ cần phối

**`find_matching_rule`** — Tìm rule phối đồ chủ đạo phù hợp nhất, fallback 3 bước.

In [ ]:
def find_matching_rule(
    user_query: str,
    gender    : str  = "female",
    profile   : dict = None,
) -> dict | None:
    """
    Tìm quy tắc phối đồ phù hợp nhất từ Layer B.

    Chiến lược fallback 3 bước:
      1. Lọc theo cả Dáng người + Tone da
      2. Nới lọc: chỉ giữ Dáng người
      3. Bỏ hết bộ lọc, lấy kết quả gần giống nhất
    """
    collection   = f"layer_b_{gender}"
    query_vector = rule_embeddings.embed_query(user_query)

    conditions = []
    if profile:
        if profile.get("dang_nguoi"):
            conditions.append(FieldCondition(
                key   = "dang_nguoi",
                match = MatchAny(any=[profile["dang_nguoi"], "Mọi vóc dáng"])
            ))
        if profile.get("tone_da"):
            conditions.append(FieldCondition(
                key   = "tone_da",
                match = MatchAny(any=[profile["tone_da"], "Mọi tone da"])
            ))

    # Bước 1: Lọc khắt khe cả Dáng + Da
    search_filter = Filter(must=conditions) if conditions else None
    response = client.query_points(
        collection_name = collection, query = query_vector,
        query_filter    = search_filter, limit = 1, score_threshold = 0.50,
    )
    results = response.points

    # Bước 2 (Fallback): Bỏ tiêu chí Da, chỉ giữ Dáng
    if not results and profile and profile.get("tone_da") and profile.get("dang_nguoi"):
        fallback_cond = [FieldCondition(
            key="dang_nguoi",
            match=MatchAny(any=[profile["dang_nguoi"], "Mọi vóc dáng"])
        )]
        response = client.query_points(
            collection_name = collection, query = query_vector,
            query_filter    = Filter(must=fallback_cond),
            limit = 1, score_threshold = 0.50,
        )
        results = response.points

    # Bước 3 (Fallback): Bỏ sạch bộ lọc
    if not results and search_filter:
        response = client.query_points(
            collection_name = collection, query = query_vector,
            limit = 1, score_threshold = 0.50,
        )
        results = response.points

    return results[0].payload if results else None

**`find_outfit_details`** — Tìm chi tiết từng món đồ (áo/quần/...) cho công thức đã chọn.

In [ ]:
def find_outfit_details(base_rule: dict, gender: str = "female") -> dict:
    """
    Từ công thức chủ đạo (đã tìm ở find_matching_rule), tìm chi tiết
    từng món đồ cần phối (VD: Áo kiểu gì, Quần kiểu gì).

    Cách 1: Khớp chính xác bằng chữ (nhanh, không cần Qdrant)
    Cách 2: Fallback semantic search Qdrant nếu không khớp
    """
    collection   = f"layer_b_{gender}"
    outfit_rules = {}
    style_query  = f"{base_rule['phong_cach']} {base_rule['boi_canh']}"

    for category in base_rule.get("goi_y_phoi_cung", []):

        # Cách 1: Khớp chính xác bằng text
        data_source = layer_b_female if gender == "female" else layer_b_male
        exact = [
            r for r in data_source
            if r["rule_key"].startswith(category)
            and r["phong_cach"] == base_rule["phong_cach"]
            and r["boi_canh"]   == base_rule["boi_canh"]
        ]
        if exact:
            outfit_rules[category] = exact[0]
            continue

        # Cách 2 (Fallback): Semantic search Qdrant
        query_vector = rule_embeddings.embed_query(f"{category} {style_query}")
        raw_response = client.query_points(
            collection_name = collection, query = query_vector,
            limit = 10, score_threshold = 0.40,
        )
        category_results = [
            r for r in raw_response.points
            if r.payload["rule_key"].startswith(category)
        ]
        if category_results:
            outfit_rules[category] = category_results[0].payload

    return outfit_rules

print("[OK] Layer B functions ready: find_matching_rule | find_outfit_details")

## PHẦN 6: Category Mapping (Layer B → Layer A)

Dịch tên category từ ngôn ngữ stylist (Layer B) sang tên kệ hàng (Layer A Qdrant).

**Ví dụ:**
- Stylist gọi `"Áo mặc trong (áo thun/sơ mi)"` → kệ hàng gọi là `["Áo"]`
- Stylist gọi `"Phụ kiện"` → phân loại thêm bằng từ khóa tiếng Anh trong `product_type`

**Tại sao cần 2 từ điển?**
- `CATEGORY_MAPPING`: dịch nhanh các loại rõ ràng
- `PHU_KIEN_KEYWORD_ROUTER`: phụ kiện rất đa dạng (mũ, kính, nhẫn…) cần soi từ khóa tiếng Anh

### Từ điển ánh xạ category Layer B → Layer A

In [ ]:
# TỪ ĐIỂN CƠ BẢN: Layer B category → Layer A shelf names
CATEGORY_MAPPING = {
    "Áo mặc trong (áo thun/sơ mi)": ["Áo"],
    "Áo khoác ngoài"              : ["Áo khoác"],
    "Áo khoác nhẹ/Áo len"         : ["Áo khoác"],
    "Quần/Chân váy"               : ["Quần", "Chân váy"],
    "Đầm/Jumpsuit"                : ["Đầm", "Jumpsuit"],
    "Giày dép"                    : ["Giày"],
    "Túi xách"                    : ["Túi xách"],
    "Phụ kiện"                    : None,  # Chung chung → dùng PHU_KIEN_KEYWORD_ROUTER
}

# TỪ ĐIỂN NÂNG CAO: soi từ khóa tiếng Anh để xác định phụ kiện cụ thể
PHU_KIEN_KEYWORD_ROUTER = {
    "Mũ"            : ["beret","hat","cap","beanie","fedora","bucket","brim","flat cap","earflap"],
    "Găng tay"       : ["gloves","glove","arm warmer"],
    "Kính mắt"       : ["glasses","sunglasses","sunglass"],
    "Đồng hồ"        : ["watch"],
    "Dây chuyền"     : ["necklace","chain pendant","chain"],
    "Bông tai"       : ["earring","earrings"],
    "Vòng tay"       : ["bracelet"],
    "Nhẫn"           : ["ring"],
    "Ghim cài áo"    : ["brooch","pin","badge"],
    "Phụ kiện hỗ trợ": ["socks","sock","scarf","tie","belt","bandana","headband","suspender"],
}

**`get_layer_a_categories`** — Phiên dịch tên category Layer B sang tên kệ hàng Layer A (riêng "Phụ kiện" sẽ soi từ khóa tiếng Anh trong `product_type`).

In [ ]:
def get_layer_a_categories(layer_b_category: str, product_type: str) -> list:
    """Phiên dịch tên category từ Layer B sang tên kệ hàng Layer A."""
    if layer_b_category != "Phụ kiện":
        return CATEGORY_MAPPING.get(layer_b_category, [])

    # Phụ kiện → soi từ khóa tiếng Anh
    ptype_lower = product_type.lower()
    for layer_a_cat, keywords in PHU_KIEN_KEYWORD_ROUTER.items():
        if any(kw in ptype_lower for kw in keywords):
            return [layer_a_cat]

    return ["Phụ kiện hỗ trợ"]

**`get_products_for_outfit`** — Tìm sản phẩm Layer A phù hợp cho từng món đồ trong outfit (phiên dịch tên kệ → similarity search → lọc theo ngưỡng score 0.20).

In [ ]:
def get_products_for_outfit(
    product_type    : str,
    layer_b_category: str,
    phong_cach      : str,
    vdb,
) -> list:
    """
    Tìm sản phẩm Layer A phù hợp cho từng món đồ trong outfit.
    Bước 1: phiên dịch tên kệ, Bước 2: similarity search, Bước 3: lọc ngưỡng 0.20.
    """
    target_categories = get_layer_a_categories(layer_b_category, product_type)

    search_filter = None
    if target_categories:
        search_filter = Filter(must=[
            FieldCondition(
                key   = "metadata.category",
                match = MatchAny(any=target_categories)
            )
        ])

    query       = f"{product_type} {phong_cach}"
    raw_results = vdb.similarity_search_with_score(query=query, k=8, filter=search_filter)

    # Ngưỡng 0.20 phù hợp cho ViFashionCLIP 512-dim (cosine similarity)
    valid_products = [
        normalize_product_metadata(doc)
        for doc, score in raw_results
        if score >= 0.20
    ]
    return diversity_filter_documents(valid_products, max_docs=3)

print("[OK] Category mapping ready: CATEGORY_MAPPING | PHU_KIEN_KEYWORD_ROUTER | get_layer_a_categories | get_products_for_outfit")

## PHẦN 7A: Vision Module — Constants & Tiền xử lý ảnh

Constants cho module Vision và helper preprocess ảnh trước khi gửi Qwen2.5-VL.

**Tại sao cần resize?**
Qwen2.5-VL có giới hạn số patch token. Ảnh >1024px có thể gây lỗi `GGML_ASSERT`.
→ Resize về `VL_MAX_SIZE=512px` (giữ tỉ lệ) trước khi encode base64.

### Hằng số Vision module (kích thước ảnh, danh sách dáng người/tone da khớp Layer B)

In [ ]:
VL_MAX_SIZE = 512   # px — giảm kích thước ảnh để tránh lỗi GGML_ASSERT

# Giá trị dáng_người khớp với Layer B (dùng để filter Qdrant)
LAYER_B_DANG_NGUOI = [
    "Dáng quả lê", "Dáng quả táo", "Dáng đồng hồ cát",
    "Dáng chữ nhật", "Dáng cân đối",
    "Người thấp bé", "Người ngoại cỡ", "Người mảnh",
]

# Giá trị tone_da khớp với Layer B
LAYER_B_TONE_DA = [
    "Da sáng", "Da trung bình", "Da ngăm", "Da ấm",
]

LAYER_B_WILDCARD_DANG = "Mọi vóc dáng"
LAYER_B_WILDCARD_TONE = "Mọi tone da"

**`_preprocess_image`** — Resize ảnh + convert JPEG base64, tránh lỗi GGML_ASSERT của Qwen-VL.

In [ ]:
def _preprocess_image(image_path: str) -> str:
    """
    Resize ảnh về VL_MAX_SIZE (giữ tỉ lệ) và convert sang JPEG base64.
    Giải quyết lỗi GGML_ASSERT: Vision encoder Qwen-VL có giới hạn patch token.
    Ảnh lớn (>1024px) vượt giới hạn → crash.
    """
    img  = Image.open(image_path).convert("RGB")
    w, h = img.size
    if max(w, h) > VL_MAX_SIZE:
        ratio = VL_MAX_SIZE / max(w, h)
        img   = img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    return base64.b64encode(buf.getvalue()).decode()

**`_call_vl`** — Gọi Qwen2.5-VL qua Ollama, không crash toàn chatbot nếu lỗi.

In [ ]:
def _call_vl(image_path: str, prompt: str) -> str:
    """
    Gọi Qwen2.5-VL qua Ollama.
    Ảnh được resize + convert JPEG trước để tránh lỗi GGML.
    Trả về chuỗi rỗng nếu có lỗi (không crash toàn bộ chatbot).
    """
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Không tìm thấy ảnh: {image_path}")
    img_b64 = _preprocess_image(image_path)
    try:
        resp = ollama.chat(
            model    = QWEN_VL_MODEL,
            messages = [{"role": "user", "content": prompt, "images": [img_b64]}],
        )
        return resp["message"]["content"].strip()
    except Exception as e:
        print(f"   [LỖI VISION] {e}")
        return ""

print(f"[OK] Vision constants:")
print(f"     Model   : {QWEN_VL_MODEL}")
print(f"     Max size: {VL_MAX_SIZE}px (auto-resize → tránh GGML_ASSERT)")
print(f"     Dáng    : {len(LAYER_B_DANG_NGUOI)} giá trị")
print(f"     Tone    : {len(LAYER_B_TONE_DA)} giá trị")

## PHẦN 7B: Vision Module — Detect / Analyze / Caption

3 functions chính của Vision Module:

| Function | Input | Output | Dùng khi |
|----------|-------|--------|----------|
| `detect_image_type` | ảnh + query | `"person"` / `"product"` | Phân loại ảnh user gửi |
| `analyze_person_image` | ảnh người | `{dang_nguoi, tone_da, nhan_xet}` | Lưu vào `user_profile` cho outfit |
| `caption_product_image` | ảnh sản phẩm | chuỗi mô tả | Fallback khi image search thất bại |

### Các tác vụ Vision cụ thể (build trên `_call_vl`)

- `detect_image_type` — xác định ảnh gửi lên là PRODUCT hay PERSON
- `analyze_person_image` — phân tích dáng người + tone da từ ảnh chân dung
- `caption_product_image` — mô tả sản phẩm trong ảnh bằng tiếng Việt (fallback khi image search rỗng)

**`detect_image_type`** — Phân loại ảnh: PRODUCT (ưu tiên) hay PERSON.

In [ ]:
def detect_image_type(image_path: str, user_query: str = "") -> str:
    """
    Xác định mục đích gửi ảnh dựa vào ảnh + câu hỏi đi kèm.
    Ưu tiên PRODUCT nếu user hỏi về sản phẩm (dù ảnh có người mẫu).
    Trả về: "product" | "person"
    """
    prompt = (
        f'Ảnh này chứa người hay chứa sản phẩm?\n'
        f'Câu hỏi của người dùng: "{user_query}"\n\n'
        f'Dựa vào câu hỏi và ảnh, xác định MỤC ĐÍCH của người dùng:\n'
        f'1. Muốn tìm/hỏi về quần áo/món đồ trong ảnh (dù có người mẫu mặc) → PRODUCT\n'
        f'2. Muốn phân tích vóc dáng/tone da của người trong ảnh             → PERSON\n'
        f'3. Không có câu hỏi + ảnh chủ yếu là người                        → PERSON\n'
        f'4. Không có câu hỏi + ảnh chụp sát sản phẩm                       → PRODUCT\n\n'
        f'Chỉ trả lời đúng 1 chữ: PERSON hoặc PRODUCT.'
    )
    result = _call_vl(image_path, prompt).upper()
    return "product" if "PRODUCT" in result else "person"

**`analyze_person_image`** — Trích xuất dáng người + tone da khớp danh sách Layer B.

In [ ]:
def analyze_person_image(image_path: str) -> dict:
    """
    Phân tích dáng người + tone da từ ảnh selfie/chân dung.
    Prompt dùng đúng các giá trị LAYER_B_DANG_NGUOI và LAYER_B_TONE_DA
    để kết quả khớp khi filter Qdrant Layer B.
    Trả về: {dang_nguoi, tone_da, nhan_xet}
    """
    dang_list = " | ".join(LAYER_B_DANG_NGUOI)
    tone_list = " | ".join(LAYER_B_TONE_DA)

    prompt = (
        f"Bạn là chuyên gia tư vấn thời trang. Phân tích người trong ảnh:\n\n"
        f"1. DÁNG NGƯỜI (chọn đúng 1 trong danh sách): {dang_list}\n"
        f"2. TONE DA (chọn đúng 1 trong danh sách): {tone_list}\n"
        f"3. NHẬN XÉT: 1-2 câu về điểm nổi bật có thể khai thác khi phối đồ.\n\n"
        f"Trả lời theo đúng format (không thêm gì khác):\n"
        f"DÁNG: [tên dáng]\n"
        f"TONE: [tên tone]\n"
        f"NHẬN XÉT: [nội dung]"
    )

    raw     = _call_vl(image_path, prompt)
    profile = {"dang_nguoi": None, "tone_da": None, "nhan_xet": ""}

    for line in raw.splitlines():
        line  = line.strip()
        upper = line.upper()
        if "DÁNG" in upper and ":" in line:
            val = line.split(":", 1)[1].strip()
            profile["dang_nguoi"] = val if val else None
        elif "TONE" in upper and ":" in line:
            val = line.split(":", 1)[1].strip()
            profile["tone_da"] = val if val else None
        elif "NHẬN XÉT" in upper and ":" in line:
            profile["nhan_xet"] = line.split(":", 1)[1].strip()

    return profile

**`caption_product_image`** — Mô tả món đồ trong ảnh, dùng làm fallback query.

In [ ]:
def caption_product_image(image_path: str, user_query: str = "") -> str:
    """
    Mô tả món đồ thời trang trong ảnh bằng tiếng Việt.
    Dùng làm fallback query khi image vector search không khả dụng.
    user_query giúp VL tập trung đúng vào món đồ user đang hỏi.
    """
    prompt = (
        f'Câu hỏi của người dùng: "{user_query}"\n\n'
        f"Mô tả chi tiết MÓN ĐỒ THỜI TRANG mà người dùng đang quan tâm trong ảnh bằng tiếng Việt.\n"
        f"Bao gồm: loại sản phẩm, màu sắc, kiểu dáng, chất liệu (nếu nhận ra), phong cách.\n"
        f"Ngắn gọn 1-2 câu. TUYỆT ĐỐI KHÔNG mô tả người mẫu."
    )
    return _call_vl(image_path, prompt)

print("[OK] Vision module ready: detect_image_type | analyze_person_image | caption_product_image")

## PHẦN 8: Khởi tạo LLM

Khởi tạo `ChatOllama` cho các tác vụ:
- Chat trả lời sản phẩm (SEARCH flow)
- Chat gợi ý outfit (OUTFIT flow)
- Phân loại intent (LLM Classifier)
- Tóm tắt lịch sử hội thoại

**Tham số quan trọng:**
- `temperature=0.4`: hơi sáng tạo, nhưng không hallucinate quá
- `num_ctx=8192`: đủ để chứa context sản phẩm + lịch sử hội thoại
- `num_predict=1024`: giới hạn output tránh phản hồi quá dài

In [ ]:
print(f"[THÔNG BÁO] Đang khởi tạo LLM qua Ollama: {LLM_MODEL}")

llm = ChatOllama(
    model       = LLM_MODEL,
    temperature = 0.4,
    timeout     = 120,
    num_predict = 1024,
    num_ctx     = 8192,
)

print(f"[OK] LLM đã sẵn sàng! Model: {LLM_MODEL}")

## PHẦN 9A: Prompt — SEARCH flow

Hai prompts cho luồng tìm kiếm sản phẩm:

1. **`system_prompt`** (`QA_PROMPT`) — prompt trả lời sản phẩm:
   - Schema bắt buộc (tên, mã, giá, đặc điểm, lý do, ảnh)
   - Chống hallucination: chỉ dùng thông tin trong `{context}`
   - Tối đa 5 sản phẩm, kết thúc bằng câu hỏi gợi mở

2. **`contextualize_q_prompt`** — rewrite câu hỏi để standalone:
   - Dựa vào lịch sử trò chuyện
   - Chỉ in câu hỏi đã rewrite, không trả lời

In [ ]:
# ── System Prompt cho SEARCH flow (schema cứng + chống hallucination) ──
system_prompt = (
    "Bạn là chuyên viên tư vấn thời trang cao cấp, thân thiện và nói tiếng Việt tự nhiên.\n\n"
    "QUY TẮC TỐI CAO:\n"
    "1. Chỉ dùng thông tin có trong phần \"DỮ LIỆU SẢN PHẨM\". Không bịa tên, mã, giá, ảnh hoặc đặc điểm.\n"
    "2. Giới thiệu tối đa 5 sản phẩm. Nếu context có nhiều hơn, chọn 5 sản phẩm phù hợp nhất.\n"
    "3. Toàn bộ câu trả lời không vượt quá 400 từ.\n"
    "4. Trước khi trả lời, tự kiểm tra sản phẩm có đúng nhu cầu không. Không trình bày quá trình suy luận.\n"
    "5. Kết thúc bằng đúng 1 câu hỏi gợi mở để tiếp tục tư vấn.\n\n"
    "SCHEMA BẮT BUỘC:\n"
    "Mình gợi ý cho bạn tối đa 5 lựa chọn sau:\n\n"
    "1. **Tên sản phẩm**\n"
    "- Mã SP: [MÃ_SP]\n"
    "- Giá: [GIÁ] VND\n"
    "- Đặc điểm: [màu/chất liệu/kiểu dáng/dịp mặc nổi bật]\n"
    "- Lý do phù hợp: [1 câu ngắn, dựa trên yêu cầu của khách]\n"
    "- Ảnh: ![Sản phẩm]([IMAGE_URL])\n\n"
    "Nếu không có ảnh, ghi: \"Ảnh: Chưa có ảnh\".\n"
    "Nếu không có sản phẩm phù hợp trong context, xin lỗi ngắn gọn và hỏi khách có muốn đổi phong cách không.\n\n"
    "DỮ LIỆU SẢN PHẨM:\n"
    "{context}"
)

QA_PROMPT = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# ── Prompt viết lại câu hỏi (query rewriting) ──
contextualize_q_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Nhiệm vụ của bạn là VIẾT LẠI CÂU HỎI.\n"
     "Dựa vào lịch sử trò chuyện, hãy làm rõ nghĩa của câu hỏi mới nhất "
     "để nó có thể đứng độc lập mà ai đọc cũng hiểu được.\n\n"
     "QUY TẮC SỐNG CÒN:\n"
     "- TUYỆT ĐỐI KHÔNG TRẢ LỜI CÂU HỎI CỦA KHÁCH.\n"
     "- CHỈ IN RA DUY NHẤT CÂU HỎI ĐÃ ĐƯỢC VIẾT LẠI. Không giải thích, không dạ thưa.\n"
     "- Nếu câu hỏi đã quá rõ ràng rồi, hãy in lại y nguyên câu đó.\n\n"
     "VÍ DỤ:\n"
     "Khách: \"Có màu khác không?\" → CHỈ IN RA: \"Áo thun đỏ ở trên có màu khác không?\""),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# ── Format document gửi vào LLM ──
doc_prompt = PromptTemplate.from_template(
    "\n[MÃ_SP: {product_id}]"
    "\nIMAGE_URL: {image_url}"
    "\nTHÔNG TIN CHI TIẾT: {page_content}\n"
)


def format_documents_for_llm(docs: list[Document]) -> str:
    """Format list of Documents thành context text cho product prompt."""
    lines = []
    for doc in docs:
        doc = normalize_product_metadata(doc)
        lines.append(doc_prompt.format(
            product_id = doc.metadata.get("product_id", "N/A"),
            image_url  = doc.metadata.get("image_url", ""),
            page_content = doc.page_content,
        ))
    return "\n".join(lines)

print("[OK] SEARCH prompts ready: QA_PROMPT | contextualize_q_prompt | doc_prompt | format_documents_for_llm")


## PHẦN 9B: Prompt — OUTFIT flow

`OUTFIT_SYSTEM_PROMPT` cho luồng gợi ý phối đồ hoàn chỉnh.

**Khác biệt so với SEARCH prompt:**
- Nhận `{outfit_context}` thay vì `{context}` — đây là chuỗi kết hợp
  công thức phối đồ Layer B + sản phẩm gợi ý Layer A
- Tối đa 3 sản phẩm (outfit gọn, không overwhelm user)
- Gắn lý do phù hợp với công thức phối đồ (không chỉ nhu cầu)

In [ ]:
OUTFIT_SYSTEM_PROMPT = (
    "Bạn là chuyên gia tạo dáng (Personal Stylist) chuyên nghiệp và tâm lý.\n\n"
    "NHIỆM VỤ: Dựa vào \"CÔNG THỨC PHỐI ĐỒ\" và \"SẢN PHẨM GỢI Ý\" bên dưới, "
    "tạo một outfit hoàn chỉnh cho khách.\n\n"
    "QUY TẮC:\n"
    "1. Chỉ giới thiệu sản phẩm có trong \"SẢN PHẨM GỢI Ý\". Không thêm món ngoài context.\n"
    "2. Tối đa 3 sản phẩm, không vượt quá 400 từ.\n"
    "3. Trước khi trả lời, tự kiểm tra sự hài hòa màu sắc, bối cảnh sử dụng và vóc dáng/tone da nếu có.\n"
    "4. Kết thúc bằng đúng 1 câu hỏi gợi mở để tiếp tục tư vấn.\n\n"
    "SCHEMA BẮT BUỘC:\n"
    "Mình phối cho bạn một set như sau:\n\n"
    "1. **Tên sản phẩm**\n"
    "- Mã SP: [MÃ_SP]\n"
    "- Giá: [GIÁ] VND\n"
    "- Đặc điểm: [màu/chất liệu/kiểu dáng nổi bật]\n"
    "- Lý do phù hợp: [1 câu ngắn, gắn với công thức phối đồ]\n"
    "- Ảnh: ![Sản phẩm]([IMAGE_URL])\n\n"
    "Nếu không có ảnh, ghi: \"Ảnh: Chưa có ảnh\".\n\n"
    "{outfit_context}"
)

outfit_prompt = ChatPromptTemplate.from_messages([
    ("system", OUTFIT_SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

print("[OK] OUTFIT prompt: OUTFIT_SYSTEM_PROMPT | outfit_prompt")

## PHẦN 9C: Redis Chat History & Summarization

Quản lý lịch sử hội thoại bằng Redis + tóm tắt tự động khi quá dài.

**Cơ chế:**
- ≤ 8 messages → giữ nguyên toàn bộ
- > 8 messages → tóm tắt phần cũ (LLM) + giữ 4 message gần nhất

**Tại sao cần tóm tắt?**
Context window của LLM có giới hạn. Lịch sử quá dài làm chậm inference
và có thể bị cắt mất thông tin quan trọng.

### Cấu hình Redis + prompt tóm tắt hội thoại

In [ ]:
REDIS_URL = "redis://localhost:6379"

SUMMARIZE_PROMPT = (
    "Tóm tắt cuộc hội thoại mua sắm thời trang sau thành 3-5 câu ngắn.\n"
    "Giữ lại: sản phẩm đã hỏi, phong cách khách thích, thông tin vóc dáng/tone da (nếu có).\n"
    "Bỏ qua: lời chào, câu xã giao.\n"
    "Chỉ trả về đoạn tóm tắt, không thêm gì khác.\n\n"
    "Hội thoại:\n"
    "{history_text}"
)

**`summarize_history`** — Dùng LLM tóm tắt các message cũ thành 3-5 câu.

In [ ]:
def summarize_history(messages: list) -> str:
    """Dùng LLM tóm tắt lịch sử hội thoại cũ."""
    history_text = "\n".join([
        f"{'Khách' if m.type == 'human' else 'Bot'}: {m.content[:300]}"
        for m in messages
    ])
    resp = ollama.chat(
        model    = LLM_MODEL,
        messages = [{"role": "user",
                     "content": SUMMARIZE_PROMPT.format(history_text=history_text)}],
        options  = {"temperature": 0, "num_predict": 150}
    )
    return resp["message"]["content"].strip()

**`get_message_history`** — Lấy lịch sử hội thoại từ Redis; nếu > 8 message thì tự tóm tắt phần cũ và chỉ giữ 4 message gần nhất (tiết kiệm context window của LLM).

In [ ]:
def get_message_history(session_id: str):
    """
    Lấy lịch sử hội thoại từ Redis.
    Tự động tóm tắt nếu > 8 messages để tiết kiệm context window.
    """
    from langchain_core.messages import SystemMessage

    history  = RedisChatMessageHistory(session_id, url=REDIS_URL)
    messages = history.messages

    # Dưới 8 message → giữ nguyên
    if len(messages) <= 8:
        return history

    # Trên 8 message → tóm tắt phần cũ, giữ 4 message gần nhất
    old_messages    = messages[:-4]
    recent_messages = messages[-4:]
    summary_text    = summarize_history(old_messages)

    history.clear()
    history.add_message(SystemMessage(
        content=f"[TÓM TẮT HỘI THOẠI TRƯỚC]: {summary_text}"
    ))
    history.add_messages(recent_messages)

    return history

print("[OK] Redis chat history ready: REDIS_URL | summarize_history | get_message_history")

## PHẦN 10: Lắp ráp RAG Pipeline

Lắp ráp 3 chains LangChain:

| Chain | Dùng cho | Input |
|-------|----------|-------|
| `full_chat_chain` | SEARCH flow đầy đủ (retriever + LLM) | `{input}` |
| `product_answer_chain_with_history` | Image search (retriever đã xong, chỉ cần LLM) | `{input, context}` |
| `outfit_chain_with_history` | OUTFIT flow | `{input, outfit_context}` |

`RunnableWithMessageHistory` bọc mỗi chain để tự động đọc/ghi Redis.

In [ ]:
print("[THÔNG BÁO] Đang lắp ráp RAG Pipeline...")

# ── Chain 1: SEARCH — full RAG (retriever + rewrite + LLM) ──
history_aware_retriever = create_history_aware_retriever(
    llm, retriever, contextualize_q_prompt
)
document_chain = create_stuff_documents_chain(
    llm=llm, prompt=QA_PROMPT, document_prompt=doc_prompt
)
rag_chain = create_retrieval_chain(history_aware_retriever, document_chain)

full_chat_chain = RunnableWithMessageHistory(
    rag_chain, get_message_history,
    input_messages_key  = "input",
    history_messages_key = "chat_history",
    output_messages_key = "answer",
)

# ── Chain 2: PRODUCT ANSWER — chỉ LLM, không gọi retriever ──
# Dùng khi image search đã lấy Documents, chỉ cần LLM viết câu trả lời
product_answer_llm_chain          = QA_PROMPT | llm
product_answer_chain_with_history = RunnableWithMessageHistory(
    product_answer_llm_chain, get_message_history,
    input_messages_key   = "input",
    history_messages_key = "chat_history",
)

# Alias để tương thích ngược với code image search cũ
image_search_chain_with_history = product_answer_chain_with_history

# ── Chain 3: OUTFIT — không gọi retriever, nhận outfit_context từ ngoài ──
outfit_llm_chain          = outfit_prompt | llm
outfit_chain_with_history = RunnableWithMessageHistory(
    outfit_llm_chain, get_message_history,
    input_messages_key   = "input",
    history_messages_key = "chat_history",
)

print("[OK] RAG Pipeline sẵn sàng: full_chat_chain | product_answer_chain_with_history | outfit_chain_with_history")


## PHẦN 11A: Intent Detection — Keyword Lists

Keyword list cho Tầng 1 (fast path) của Hybrid Intent Detection.

**Nguyên tắc chọn keyword:**
- Chỉ dùng từ **không bao giờ nhầm intent** (precision ~100%)
- Outfit/Search ưu tiên hơn Greeting/Chitchat
- Từ "anh" bị loại khỏi MALE_KEYWORDS vì "anh ơi" là cách user nữ gọi chatbot

In [ ]:
# ── Tầng 1: Keyword list "đảm bảo" (không bao giờ nhầm intent) ──
DEFINITE_GREETING = [
    "xin chào", "hello", "hi bạn", "chào bạn", "hey", "alo",
    "chào buổi sáng", "chào buổi chiều",
]
DEFINITE_CHITCHAT = [
    "cảm ơn", "cảm on", "thank you", "thanks", "tạm biệt",
    "bye", "hẹn gặp lại", "bái bai",
]
DEFINITE_OUTFIT = [
    "phối đồ", "mix match", "mặc với gì", "mặc cùng gì",
    "kết hợp với gì", "phối với gì", "outfit cho",
    "gợi ý outfit", "tư vấn phối",
]
DEFINITE_SEARCH = [
    "còn hàng không", "còn size", "giá bao nhiêu", "mã sp",
    "có bán không", "tìm giúp", "cho xem", "shop có",
    "xem thêm", "xem them", "cho xem thêm", "cho xem them",
]

# Từ khóa giới tính nam — chỉ dùng từ rõ ràng, tránh false positive
# Lưu ý: "anh" bị loại khỏi list vì hay xuất hiện trong "anh ơi" (user nữ gọi bot)
MALE_KEYWORDS = ["nam", "con trai", "bạn trai", "chàng", "đàn ông", "bố", "chồng"]

print("[OK] Intent keyword lists: DEFINITE_GREETING | DEFINITE_CHITCHAT | DEFINITE_OUTFIT | DEFINITE_SEARCH | MALE_KEYWORDS")

## PHẦN 11B: Intent Detection — LLM Classifier & Hybrid

**Tầng 2:** LLM classify cho câu mơ hồ mà keyword không xử lý được.

**`detect_intent_llm()`** — gọi LLM với context bot message trước để hiểu đúng hơn.
VD: user trả lời "đi làm" khi bot hỏi "dịp nào" → intent là "outfit", không phải chitchat.

**`detect_intent()`** — Hybrid:
1. Keyword (nhanh, không gọi LLM)
2. LLM (chậm hơn, chính xác hơn cho câu mơ hồ)

### Prompt phân loại intent bằng LLM (Tầng 2 — dùng khi keyword không khớp)

In [ ]:
INTENT_CLASSIFY_PROMPT = (
    "Bạn là bộ phân loại intent cho chatbot tư vấn thời trang.\n"
    "Phân loại câu hỏi vào đúng 1 trong 5 nhóm:\n\n"
    "OUTFIT   → Hỏi cách phối đồ, mix-match, tư vấn mặc gì cho dịp/vóc dáng/phong cách\n"
    "SEARCH   → Tìm sản phẩm cụ thể, hỏi giá, còn hàng, so sánh, xem ảnh sản phẩm\n"
    "CHITCHAT → Cảm ơn, tạm biệt, hỏi thăm, câu xã giao không liên quan mua sắm\n"
    "GREETING → Chào hỏi, bắt đầu cuộc trò chuyện\n\n"
    "Ví dụ bắt buộc phải học thuộc:\n"
    "\"có áo nào mặc đi tiệc không?\"          → SEARCH   (tìm sản phẩm, dù có từ \"mặc\")\n"
    "\"tìm đồ phù hợp với quần jean xanh\"     → OUTFIT   (phối đồ)\n"
    "\"tôi cao 1m55 nên mặc gì?\"              → OUTFIT   (tư vấn theo vóc dáng)\n"
    "\"còn size L không?\"                     → SEARCH   (hỏi tồn kho)\n"
    "\"oke cảm ơn bạn nhiều\"                  → CHITCHAT\n"
    "\"đi làm\"                                → OUTFIT   (trả lời câu hỏi \"dịp nào\" của bot)\n"
    "{context_block}\n"
    "Câu cần phân loại: \"{query}\"\n\n"
    "Chỉ trả lời đúng 1 từ: OUTFIT / SEARCH / CHITCHAT / GREETING"
)

**`detect_intent_llm`** — Gọi LLM phân loại intent cho câu hỏi mơ hồ.

In [ ]:
def detect_intent_llm(query: str, last_bot_msg: str = "") -> str:
    """
    Dùng LLM phân loại intent cho câu mơ hồ.
    Truyền thêm last_bot_msg để LLM hiểu context (VD: user trả lời câu hỏi của bot).
    Fallback an toàn: "search" nếu LLM lỗi.
    """
    context_block = ""
    if last_bot_msg:
        context_block = f'\nContext — Bot vừa nói: "{last_bot_msg[:120]}..."\n'

    try:
        resp = ollama.chat(
            model    = LLM_MODEL,
            messages = [{"role": "user",
                         "content": INTENT_CLASSIFY_PROMPT.format(
                             query         = query,
                             context_block = context_block,
                         )}],
            options  = {
                "temperature": 0,   # deterministic
                "num_predict": 10,  # chỉ cần 1 từ → rất nhanh (~0.3-0.5s)
            }
        )
        result = resp["message"]["content"].strip().upper()
        for intent in ["OUTFIT", "SEARCH", "CHITCHAT", "GREETING"]:
            if intent in result:
                return intent.lower()
    except Exception as e:
        print(f"[WARN] LLM intent classify lỗi: {e} → fallback search")

    return "search"   # fallback an toàn

**`detect_intent`** — Hybrid: Tầng 1 keyword "đảm bảo" (nhanh, không gọi LLM) → Tầng 2 LLM classify nếu không khớp keyword nào.

**`detect_gender`** — Suy luận giới tính khách hàng từ câu hỏi (mặc định female).

In [ ]:
def detect_intent(query: str, last_bot_msg: str = "") -> str:
    """
    Hybrid intent detection:
      Tầng 1 — Keyword "đảm bảo": trả về ngay (không gọi LLM)
      Tầng 2 — LLM classify: câu mơ hồ + context bot message trước

    Thứ tự ưu tiên: outfit > search > greeting > chitchat
    """
    q = query.lower().strip()

    if any(kw in q for kw in DEFINITE_OUTFIT):   return "outfit"
    if any(kw in q for kw in DEFINITE_SEARCH):   return "search"
    if any(kw in q for kw in DEFINITE_GREETING): return "greeting"
    if any(kw in q for kw in DEFINITE_CHITCHAT): return "chitchat"

    return detect_intent_llm(query, last_bot_msg)

## PHẦN 11C: Intent Detection — Simple Response Handlers

Response cố định cho intent `greeting` và `chitchat` — không cần gọi LLM,
trả về ngay để giảm latency.

In [ ]:
def get_greeting_response() -> str:
    """Response cố định cho lời chào."""
    return (
        "Xin chào! Mình là trợ lý tư vấn thời trang của shop. "
        "Bạn cần tìm sản phẩm hay muốn được gợi ý phối đồ hôm nay? 😊"
    )


def get_chitchat_response(query: str) -> str:
    """Response cố định cho chitchat (cảm ơn, tạm biệt)."""
    return "Rất vui được hỗ trợ bạn! Bạn còn muốn hỏi thêm gì về thời trang không?"


print("[OK] Simple handlers: get_greeting_response | get_chitchat_response")

## PHẦN 11D: Intent Detection — Test Cases

Kiểm tra nhanh Tầng 1 (keyword) của intent detection — không gọi LLM.

**Kết quả mong đợi:**
- ✅ keyword khớp đúng intent
- 🔁 câu mơ hồ → cần LLM (chạy `detect_intent()` thực sự để kiểm tra Tầng 2)

In [ ]:
TEST_CASES = [
    ("xin chào bạn",                        "greeting"),
    ("cảm ơn bạn nhiều nha",                "chitchat"),
    ("phối đồ với áo thun trắng",           "outfit"),
    ("còn size L không",                    "search"),
    ("tôi cao 1m55 nên mặc gì?",            "outfit → LLM"),
    ("có áo nào mặc đi tiệc không?",        "search → LLM"),
    ("muốn trông thanh lịch hơn thì sao?",  "outfit → LLM"),
    ("đi làm",                              "outfit → LLM (cần context)"),
]

print("--- Kiểm tra Tầng 1 (không gọi LLM) ---")
for query, expected in TEST_CASES:
    q     = query.lower().strip()
    tier1 = None
    if any(kw in q for kw in DEFINITE_OUTFIT):    tier1 = "outfit"
    elif any(kw in q for kw in DEFINITE_SEARCH):  tier1 = "search"
    elif any(kw in q for kw in DEFINITE_GREETING): tier1 = "greeting"
    elif any(kw in q for kw in DEFINITE_CHITCHAT): tier1 = "chitchat"

    if tier1:
        status = "✅" if tier1 == expected.split(" →")[0] else "⚠️"
        print(f"  {status} [{tier1:8s}] ← keyword | '{query}'")
    else:
        print(f"  🔁 [ambiguous] → cần LLM  | '{query}' (expect: {expected})")

## PHẦN 12: Outfit Context Builder (Layer B → Layer A)

`build_outfit_context()` — xâu chuỗi Layer B và Layer A thành hồ sơ outfit hoàn chỉnh.

**Luồng:**
```
user_query + gender + profile
    ↓
find_matching_rule()          → base_rule (công thức phối đồ)
    ↓
find_outfit_details()         → outfit_rules (chi tiết từng món)
    ↓
get_products_for_outfit()     → outfit_products (sản phẩm Layer A)
    ↓
format text → CÔNG THỨC + SẢN PHẨM GỢI Ý → outfit_context string
```

Nếu Layer B không khớp → trả về `""` → Chat loop fallback sang SEARCH flow.

In [ ]:
def build_outfit_context(
    user_query: str,
    gender    : str  = "female",
    profile   : dict = None,
) -> str:
    """
    Xâu chuỗi Layer B và Layer A thành hồ sơ outfit hoàn chỉnh cho LLM.
    Trả về chuỗi text có cấu trúc, hoặc "" nếu không tìm thấy rule.
    """
    base_rule = find_matching_rule(user_query, gender, profile)
    if not base_rule:
        return ""

    outfit_rules = find_outfit_details(base_rule, gender)
    if not outfit_rules:
        return ""

    outfit_products = {}
    for layer_b_category, rule in outfit_rules.items():
        product_type = rule["rule_key"].split("|")[1].strip()
        products     = get_products_for_outfit(
            product_type, layer_b_category, base_rule["phong_cach"], vector_db
        )
        outfit_products[layer_b_category] = {
            "product_type": product_type,
            "ly_do"       : rule["ly_do_tu_van"],
            "products"    : products,
        }

    lines = [
        "CÔNG THỨC PHỐI ĐỒ:",
        f"  Phong cách: {base_rule['phong_cach']}",
        f"  Bối cảnh : {base_rule['boi_canh']}",
        f"  Lý do    : {base_rule['ly_do_tu_van']}",
    ]
    if profile and profile.get("dang_nguoi"):
        lines.append(f"  Dáng người: {profile['dang_nguoi']}")
    if profile and profile.get("tone_da"):
        lines.append(f"  Tone da   : {profile['tone_da']}")
    lines += ["", "SẢN PHẨM GỢI Ý:"]

    total_products = 0
    for cat, data in outfit_products.items():
        lines.append(f"\n[{cat} — {data['product_type']}]")
        lines.append(f"  Lý do: {data['ly_do']}")
        if data["products"]:
            for doc in data["products"]:
                if total_products >= 3:
                    break
                doc       = normalize_product_metadata(doc)
                pid       = doc.metadata.get("product_id", "N/A")
                price_raw = doc.metadata.get("price", "N/A")
                image_url = doc.metadata.get("image_url", "")
                try:
                    price_fmt = f"{int(price_raw):,}".replace(",", ".")
                except Exception:
                    price_fmt = price_raw
                lines.append(f"  - (Mã SP: {pid} | Giá: {price_fmt} VND | IMAGE_URL: {image_url})")
                lines.append(f"    {doc.page_content[:600]}")
                total_products += 1
        else:
            lines.append("  - (Chưa có sản phẩm phù hợp trong kho)")
        if total_products >= 3:
            break

    return "\n".join(lines)


print("[OK] build_outfit_context sẵn sàng (tối đa 3 sản phẩm + image_url)")


## PHẦN 13A: Input Validation & Security

`validate_user_query()` chặn 2 loại input độc hại trước khi vào intent/RAG:

1. **Input quá dài** (> 500 ký tự): tránh context injection và tăng cost
2. **Prompt injection** (regex pattern): các mẫu như "ignore previous instructions", "đóng vai", "tiết lộ prompt"

**Tại sao xử lý ở lớp input thay vì system prompt?**
System prompt không đảm bảo 100% — regex filter là lớp bảo vệ cứng, không thể bị "thuyết phục".

### Chống prompt injection & giới hạn độ dài input

`MAX_QUERY_CHARS` + `PROMPT_INJECTION_PATTERNS` (mỗi pattern có cả bản không dấu và có dấu, phòng trường hợp user gõ tiếng Việt không dấu để né filter).

In [ ]:
MAX_QUERY_CHARS = 500

# Các pattern prompt injection phổ biến (tiếng Việt + tiếng Anh)
PROMPT_INJECTION_PATTERNS = [
    r"ignore\s+(all\s+)?previous",
    r"bo\s+qua\s+(moi\s+)?(huong\s+dan|lenh|prompt)",
    r"b\u1ecf\s+qua\s+(m\u1ecdi\s+)?(h\u01b0\u1edbng\s+d\u1eabn|l\u1ec7nh|prompt)",
    r"quen\s+(het|cac)\s+(huong\s+dan|lenh)",
    r"qu\u00ean\s+(h\u1ebft|c\u00e1c)\s+(h\u01b0\u1edbng\s+d\u1eabn|l\u1ec7nh)",
    r"dong\s+vai",
    r"\u0111\u00f3ng\s+vai",
    r"system\s+prompt",
    r"developer\s+message",
    r"tiet\s+lo\s+(prompt|huong\s+dan|lenh)",
    r"ti\u1ebft\s+l\u1ed9\s+(prompt|h\u01b0\u1edbng\s+d\u1eabn|l\u1ec7nh)",
]

**`validate_user_query`** — Kiểm tra độ dài + prompt injection trước khi xử lý.

In [ ]:
def validate_user_query(query: str) -> tuple[bool, str]:
    """
    Validate input trước khi xử lý.
    Trả về (True, "") nếu hợp lệ, hoặc (False, error_message) nếu vi phạm.
    """
    clean_query = query.strip()
    if len(clean_query) > MAX_QUERY_CHARS:
        return False, f"Tin nhắn hơi dài rồi ạ. Bạn rút gọn dưới {MAX_QUERY_CHARS} ký tự giúp mình nhé."
    lowered = clean_query.lower()
    if any(re.search(p, lowered, flags=re.IGNORECASE) for p in PROMPT_INJECTION_PATTERNS):
        return False, "Mình chỉ hỗ trợ tư vấn thời trang và sản phẩm trong shop. Bạn gửi lại nhu cầu mua sắm cụ thể giúp mình nhé."
    return True, ""

**`SpyRetrieverHandler`** — Callback debug: in ra câu hỏi sau khi LLM rewrite (query rewriting).

In [ ]:
class SpyRetrieverHandler(BaseCallbackHandler):
    """Hiển thị câu hỏi sau khi LLM đã rewrite (dùng để debug retrieval)."""
    def on_retriever_start(self, serialized: dict, query: str, **kwargs):
        print(f"\n[DEBUG] Câu hỏi sau rewrite: {query}\n")


print("[OK] Security layer: validate_user_query | SpyRetrieverHandler")
print(f"     Max input: {MAX_QUERY_CHARS} chars | Patterns: {len(PROMPT_INJECTION_PATTERNS)}")

## PHẦN 13B: Chat Loop chính

Vòng lặp chat chính của chatbot. Xử lý tuần tự:

```
1. Đọc input text
2. Validate (độ dài + injection)
3. Đọc input ảnh (nếu có)
   ├── PERSON → analyze_person_image → lưu user_profile → tiếp tục
   └── PRODUCT → search_products_by_image (FashionCLIP) hoặc caption (fallback)
4. Validate final_query
5. Detect intent + gender
6. Route đến handler:
   ├── greeting/chitchat → response cố định
   ├── image_search → product_answer_chain_with_history
   ├── outfit → build_outfit_context → outfit_chain_with_history
   └── search → full_chat_chain (full RAG)
7. Stream output + in metrics (TTFT, total time)
```

Nhập `0` để thoát.

### Vòng lặp chat chính

Đây là entry point chạy chatbot trong terminal/notebook. Toàn bộ 7 bước xử lý mỗi lượt chat được liệt kê trong comment đầu code cell bên dưới.

In [ ]:
# ════════════════════════════════════════════════════════════════
# VÒNG LẶP CHAT CHÍNH (Chat Loop)
#
# Luồng xử lý mỗi lượt chat:
#   1. Validate input text (chống prompt injection, giới hạn ký tự)
#   2. Nếu có ảnh đính kèm → phân loại PERSON / PRODUCT rồi xử lý tương ứng
#   3. Validate lại câu hỏi cuối cùng (sau khi đã gộp caption ảnh nếu có)
#   4. Detect intent (greeting / chitchat / search / outfit / image_search) + giới tính
#   5. Trả lời nhanh cho intent đơn giản (greeting/chitchat), không cần gọi LLM chính
#   6. Gọi đúng RAG chain tương ứng với intent, stream token ra màn hình
#   7. In metrics: TTFT (time-to-first-token) và tổng thời gian phản hồi
# ════════════════════════════════════════════════════════════════

SESSION_ID   = str(uuid.uuid4())
user_profile = {}  # Tích lũy dáng người + tone da qua từng lượt chat

print("=" * 60)
print("  CHATBOT TƯ VẤN THỜI TRANG  ")
print("  Nhập '0' để thoát | Nhập đường dẫn ảnh nếu có")
print("=" * 60 + "\n")

while True:
    user_input = input("Bạn: ").strip()
    if user_input == "0":
        print("\nChatbot: Cảm ơn bạn đã ghé thăm shop. Hẹn gặp lại!")
        break
    if not user_input:
        continue

    # ── 1. Validate text input ──────────────────────────────────────
    is_valid, validation_message = validate_user_query(user_input)
    if not is_valid:
        print(f"Chatbot: {validation_message}")
        print("\n" + "-" * 60 + "\n")
        continue

    # ── 2. Xử lý ảnh (nếu có) ────────────────────────────────────────
    force_image_search = False
    image_search_docs  = []
    final_query         = user_input

    raw_img = input("Ảnh (Enter để bỏ qua): ").strip()
    if raw_img:
        if not os.path.exists(raw_img):
            print(f"   Không tìm thấy file: {raw_img}")
        else:
            print("[Đang phân tích ảnh...]")
            image_type = detect_image_type(raw_img, user_input)
            print(f"   -> Phát hiện: {image_type.upper()}")

            if image_type == "person":
                # Phân tích dáng người + tone da -> lưu vào user_profile
                person_info = analyze_person_image(raw_img)
                if person_info["dang_nguoi"]:
                    user_profile["dang_nguoi"] = person_info["dang_nguoi"]
                if person_info["tone_da"]:
                    user_profile["tone_da"] = person_info["tone_da"]

                print(
                    f"\nChatbot: Mình đã phân tích xong! "
                    f"Bạn có dáng **{person_info['dang_nguoi']}** "
                    f"với **{person_info['tone_da']}**. "
                    f"{person_info['nhan_xet']} "
                    f"\n\nMình đã lưu thông tin này để tư vấn phối đồ phù hợp hơn. "
                    f"Bạn muốn gợi ý outfit cho dịp nào?"
                )
                print("\n" + "-" * 60 + "\n")
                continue

            # Ảnh sản phẩm -> FashionCLIP image search
            image_search_docs = search_products_by_image(
                raw_img,
                top_k        = PRODUCT_SEARCH_CANDIDATE_K,
                max_products = PRODUCT_SEARCH_PAGE_SIZE,
            )
            if image_search_docs:
                force_image_search = True
                final_query = f"Find products similar to the uploaded image. Extra request: {user_input}"
                print(f"   -> FashionCLIP tìm thấy {len(image_search_docs)} sản phẩm tương tự")
            else:
                print("   -> Image vector search không khả dụng, dùng Qwen-VL caption làm fallback")
                caption = caption_product_image(raw_img, user_input)
                print(f"   -> Caption: {caption[:80]}...")
                final_query = f"{caption}. Extra request: {user_input}" if user_input else caption

    # ── 3. Validate lại final_query (sau khi gộp caption ảnh) ────────
    is_valid, validation_message = validate_user_query(final_query)
    if not is_valid:
        print(f"Chatbot: {validation_message}")
        print("\n" + "-" * 60 + "\n")
        continue

    # ── 4. Detect intent + giới tính ──────────────────────────────────
    intent         = "image_search" if force_image_search else detect_intent(final_query)
    current_gender = detect_gender(final_query)
    if current_gender == "male":
        user_profile["gender"] = "male"
    gender = user_profile.get("gender", current_gender)

    profile_status = (
        f"Giới: {gender.upper()} | "
        f"Dáng: {user_profile.get('dang_nguoi', '?')} | "
        f"Tone: {user_profile.get('tone_da', '?')}"
    ) if user_profile else "chưa có"
    print(f"[Intent: {intent.upper()} | Profile: {profile_status}]")

    # ── 5. Simple intents (không cần LLM chính) ───────────────────────
    if intent == "greeting":
        print(f"Chatbot: {get_greeting_response()}")
        print("\n" + "-" * 60 + "\n")
        continue

    if intent == "chitchat":
        print(f"Chatbot: {get_chitchat_response(final_query)}")
        print("\n" + "-" * 60 + "\n")
        continue

    # ── 6. Gọi LLM chain tương ứng, stream token ────────────────────
    print("Chatbot: ", end="")
    start_time       = time.time()
    first_token_time = None

    try:
        if intent == "image_search":
            image_context = format_documents_for_llm(image_search_docs)
            for chunk in product_answer_chain_with_history.stream(
                {"input": final_query, "context": image_context},
                config={"configurable": {"session_id": SESSION_ID}},
            ):
                token = chunk.content if hasattr(chunk, "content") else str(chunk)
                if token:
                    if first_token_time is None: first_token_time = time.time()
                    print(token, end="", flush=True)

        if intent == "outfit":
            outfit_context = build_outfit_context(final_query, gender, user_profile)
            if not outfit_context:
                print("[Layer B không khớp -> fallback RAG search]")
                intent = "search"
            else:
                for chunk in outfit_chain_with_history.stream(
                    {"input": user_input, "outfit_context": outfit_context},
                    config={"configurable": {"session_id": SESSION_ID}},
                ):
                    token = chunk.content if hasattr(chunk, "content") else str(chunk)
                    if token:
                        if first_token_time is None: first_token_time = time.time()
                        print(token, end="", flush=True)

        if intent == "search":
            for chunk in full_chat_chain.stream(
                {"input": final_query},
                config={
                    "configurable": {"session_id": SESSION_ID},
                    "callbacks"   : [SpyRetrieverHandler()],
                },
            ):
                if "answer" in chunk:
                    if first_token_time is None: first_token_time = time.time()
                    print(chunk["answer"], end="", flush=True)

        # ── 7. Metrics ────────────────────────────────────────────────
        end_time = time.time()
        if first_token_time is None: first_token_time = end_time
        print(f"\n\nTTFT: {first_token_time - start_time:.2f}s | Tổng: {end_time - start_time:.2f}s")

    except Exception as e:
        print(f"\n[LỖI] {e}")

    print("\n\n" + "-" * 60 + "\n")
